# CACEIS Human Capital KPI Technical Notebook

This notebook embeds all 5 KPI pipeline codes directly (no imports from `kpi1_pipeline.py` to `kpi5_pipeline.py`).

Notebook goals:
1. Explain what each KPI code does and why.
2. Run each pipeline end-to-end on the CACEIS datasets.
3. Check the technical deliverable checklist from `todo.txt`.


In [23]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)


In [24]:
# KPI 1 Pipeline Code
def _register_kpi1_pipeline():

    import re
    from dataclasses import dataclass
    from pathlib import Path
    from typing import Literal

    import pandas as pd

    Scope = Literal["group", "europe"]


    @dataclass(frozen=True)
    class KPI1Config:
        file_path: Path
        scope: Scope = "group"


    PL_ROW_PNB = "Net Banking Income (PNB)"
    PL_ROW_OTHER_OPERATING_COSTS = "Other Operating Costs"
    PL_ROW_TOTAL_PERSONNEL_COSTS = "Total Personnel Costs"


    def _read_pl_raw(file_path: Path) -> pd.DataFrame:
        return pd.read_excel(file_path, sheet_name="Synthese_PL")


    def _read_fte_raw(file_path: Path) -> pd.DataFrame:
        return pd.read_excel(file_path, sheet_name="Synthese_ETP", header=None)


    def _extract_year(text: object) -> int | None:
        if text is None:
            return None
        match = re.search(r"(\d{4})", str(text))
        return int(match.group(1)) if match else None


    def extract_pl_rows(file_path: Path, scope: Scope = "group") -> pd.DataFrame:
        """
        Step 1 in KPI_PLAN:
        Extract yearly P&L rows from Synthese_PL.
        """
        raw = _read_pl_raw(file_path)

        if scope == "group":
            label_col = raw.columns[0]
            value_cols = raw.columns[1:5]
        elif scope == "europe":
            label_col = raw.columns[6]
            value_cols = raw.columns[7:11]
        else:
            raise ValueError(f"Unsupported scope: {scope}")

        table = raw.loc[:, [label_col, *value_cols]].copy()
        year_cols = [2022, 2023, 2024, 2025]
        table.columns = ["metric", *year_cols]

        table["metric"] = table["metric"].astype(str).str.strip()
        table = table[table["metric"].isin(
            {
                PL_ROW_PNB,
                PL_ROW_OTHER_OPERATING_COSTS,
                PL_ROW_TOTAL_PERSONNEL_COSTS,
            }
        )]

        long = table.melt(
            id_vars="metric",
            value_vars=year_cols,
            var_name="year",
            value_name="value_raw",
        )
        long["value_raw"] = pd.to_numeric(long["value_raw"], errors="coerce")
        long = long.dropna(subset=["value_raw"])

        # In the source sheet, cost lines are negative; convert to positive costs for business KPIs.
        long["value"] = long["value_raw"]
        cost_mask = long["metric"].isin(
            [PL_ROW_OTHER_OPERATING_COSTS, PL_ROW_TOTAL_PERSONNEL_COSTS]
        )
        long.loc[cost_mask, "value"] = long.loc[cost_mask, "value_raw"].abs()

        return long.sort_values(["year", "metric"]).reset_index(drop=True)


    def extract_avg_fte(file_path: Path, scope: Scope = "group") -> pd.DataFrame:
        """
        Step 2 in KPI_PLAN:
        Extract yearly average FTE from Synthese_ETP.
        """
        raw = _read_fte_raw(file_path)

        if scope == "group":
            row_mask = raw.iloc[:, 0].astype(str).str.contains(
                "TOTAL \\(Toutes Filiales conso\\)", regex=True, na=False
            )
        elif scope == "europe":
            row_mask = raw.iloc[:, 0].astype(str).str.contains(
                "o/w TOTAL \\(Europe\\)", regex=True, na=False
            )
        else:
            raise ValueError(f"Unsupported scope: {scope}")

        if not row_mask.any():
            raise ValueError(f"Unable to locate FTE row for scope={scope!r}")

        row_values = raw.loc[row_mask].iloc[0]

        records: list[dict[str, float | int]] = []
        for col_idx in raw.columns:
            second_header = str(raw.iat[1, col_idx]).strip()
            if second_header != "ETP moyen période":
                continue

            year = _extract_year(raw.iat[0, col_idx])
            if year is None and isinstance(col_idx, int) and col_idx > 0:
                # In this sheet layout, year labels can be stored in the previous column.
                year = _extract_year(raw.iat[0, col_idx - 1])
            if year is None:
                continue

            value = pd.to_numeric(row_values[col_idx], errors="coerce")
            if pd.notna(value):
                records.append({"year": int(year), "avg_fte": float(value)})

        if not records:
            raise ValueError("No 'ETP moyen période' columns found in Synthese_ETP")

        return pd.DataFrame(records).sort_values("year").reset_index(drop=True)


    def build_kpi1_table(config: KPI1Config) -> pd.DataFrame:
        """
        Step 3 in KPI_PLAN:
        Build yearly finance KPI table, 2022-2025.
        """
        pl_long = extract_pl_rows(config.file_path, config.scope)
        fte = extract_avg_fte(config.file_path, config.scope)

        pivot = pl_long.pivot(index="year", columns="metric", values="value").reset_index()
        pivot.columns.name = None

        kpi = pivot.merge(fte, on="year", how="inner")

        kpi["hcva"] = kpi[PL_ROW_PNB] - kpi[PL_ROW_OTHER_OPERATING_COSTS]
        kpi["hcva_per_fte"] = kpi["hcva"] / kpi["avg_fte"]
        kpi["hcroi"] = kpi["hcva"] / kpi[PL_ROW_TOTAL_PERSONNEL_COSTS]
        kpi["revenue_per_fte"] = kpi[PL_ROW_PNB] / kpi["avg_fte"]

        ordered_cols = [
            "year",
            PL_ROW_PNB,
            PL_ROW_OTHER_OPERATING_COSTS,
            PL_ROW_TOTAL_PERSONNEL_COSTS,
            "avg_fte",
            "hcva",
            "hcva_per_fte",
            "hcroi",
            "revenue_per_fte",
        ]
        return kpi.loc[:, ordered_cols].sort_values("year").reset_index(drop=True)


    def summarize_trend(kpi_table: pd.DataFrame) -> list[str]:
        """
        Step 5 in KPI_PLAN:
        Provide business interpretation for KPI movement over time.
        """
        if kpi_table.empty or len(kpi_table) < 2:
            return ["Not enough yearly points to compute trend interpretation."]

        start = kpi_table.iloc[0]
        end = kpi_table.iloc[-1]

        def pct_change(start_value: float, end_value: float) -> float:
            if start_value == 0:
                return float("nan")
            return ((end_value - start_value) / start_value) * 100.0

        hcva_fte_change = pct_change(start["hcva_per_fte"], end["hcva_per_fte"])
        hcroi_change = pct_change(start["hcroi"], end["hcroi"])
        rev_fte_change = pct_change(start["revenue_per_fte"], end["revenue_per_fte"])

        lines = [
            (
                f"HCVA per FTE moved from {start['hcva_per_fte']:,.2f} in {int(start['year'])} "
                f"to {end['hcva_per_fte']:,.2f} in {int(end['year'])} ({hcva_fte_change:+.1f}%)."
            ),
            (
                f"HCROI moved from {start['hcroi']:.2f} to {end['hcroi']:.2f} "
                f"({hcroi_change:+.1f}%)."
            ),
            (
                f"Revenue per FTE moved from {start['revenue_per_fte']:,.2f} "
                f"to {end['revenue_per_fte']:,.2f} ({rev_fte_change:+.1f}%)."
            ),
        ]

        if end["hcroi"] >= start["hcroi"] and end["hcva_per_fte"] >= start["hcva_per_fte"]:
            lines.append(
                "Interpretation: workforce value creation improved relative to workforce cost."
            )
        elif end["hcroi"] < start["hcroi"] and end["hcva_per_fte"] < start["hcva_per_fte"]:
            lines.append(
                "Interpretation: value creation is weakening versus workforce cost; investigate drivers."
            )
        else:
            lines.append(
                "Interpretation: signals are mixed; inspect entity mix, cost structure, and productivity drivers."
            )

        return lines

    return KPI1Config, build_kpi1_table, summarize_trend

KPI1Config, build_kpi1_table, summarize_trend = _register_kpi1_pipeline()


In [25]:
# KPI 2 Pipeline Code
def _register_kpi2_pipeline():

    from dataclasses import dataclass
    from pathlib import Path
    from typing import Iterable

    import pandas as pd


    DEFAULT_PANEL_PATH = Path("Sujet Alberthon/HR Data/Data.xlsx")
    DEFAULT_EAE_PATH = Path(
        "Sujet Alberthon/HR Data/20250218 - Stats CACEIS EAE EP 18-02-2025 Version Définitive cloture.xlsx"
    )
    DEFAULT_TRAINING_PATH = Path("Sujet Alberthon/Training/Training_Records_Unnamed.xlsx")
    DEFAULT_ABSENCE_PATHS = (
        Path("Sujet Alberthon/HR Data/Absentéisme_-_détail_affectation_-_Bilan_social.xlsx"),
        Path("Sujet Alberthon/HR Data/Absentéisme_-_détail_affectation_-_Bilan_social (1).xlsx"),
        Path(
            "Sujet Alberthon/HR Data/20260121 - Absentéisme_-_détail_affectation_-_Bilan_social 2025.xlsx"
        ),
    )


    @dataclass(frozen=True)
    class KPI2Config:
        panel_path: Path = DEFAULT_PANEL_PATH
        eae_path: Path = DEFAULT_EAE_PATH
        training_path: Path = DEFAULT_TRAINING_PATH
        absence_paths: tuple[Path, ...] = DEFAULT_ABSENCE_PATHS


    @dataclass
    class KPI2Result:
        risk_table: pd.DataFrame
        latest_risk_table: pd.DataFrame
        entity_risk: pd.DataFrame
        job_risk: pd.DataFrame
        segment_risk: pd.DataFrame
        risk_trend: pd.DataFrame


    def _normalize_id(values: pd.Series) -> pd.Series:
        cleaned = values.astype(str).str.strip()
        return cleaned.replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA, "None": pd.NA, "NaT": pd.NA})


    def _to_month_start(values: pd.Series) -> pd.Series:
        parsed = pd.to_datetime(values, errors="coerce", dayfirst=True)
        return parsed.dt.to_period("M").dt.to_timestamp()


    def _months_diff(start: pd.Series, end: pd.Series) -> pd.Series:
        delta = (end.dt.year - start.dt.year) * 12 + (end.dt.month - start.dt.month)
        delta = delta.where(start.notna() & end.notna())
        return delta.clip(lower=0)


    def build_employee_month_table(panel_path: Path) -> pd.DataFrame:
        """
        KPI_PLAN step 1:
        Build employee-month table from Data.xlsx > Sheet1.
        """
        raw = pd.read_excel(panel_path, sheet_name="Sheet1")

        use_cols = [
            "COUNTRY_GROUP",
            "COUNTRY_GROUP_LABEL_EN",
            "PERIOD",
            "ID Employee",
            "Age range",
            "SEXE_GROUP_LABEL_EN",
            "CONTRACT_GROUP_LABEL_EN",
            "DATE_ENTRY_CACEIS",
            "DATE_ENTRY_POSTE",
            "POSTE_LABEL_LOCAL",
            "ENTITY_LABEL_LOCAL",
        ]

        missing_cols = [col for col in use_cols if col not in raw.columns]
        if missing_cols:
            raise ValueError(f"Missing columns in panel source: {missing_cols}")

        panel = raw.loc[:, use_cols].rename(
            columns={
                "COUNTRY_GROUP": "country_code",
                "COUNTRY_GROUP_LABEL_EN": "country",
                "PERIOD": "period",
                "ID Employee": "employee_id",
                "Age range": "age_range",
                "SEXE_GROUP_LABEL_EN": "gender",
                "CONTRACT_GROUP_LABEL_EN": "contract_type",
                "DATE_ENTRY_CACEIS": "date_entry_caceis",
                "DATE_ENTRY_POSTE": "date_entry_poste",
                "POSTE_LABEL_LOCAL": "job",
                "ENTITY_LABEL_LOCAL": "entity",
            }
        )

        panel["employee_id"] = _normalize_id(panel["employee_id"])
        panel["period"] = _to_month_start(panel["period"])
        panel["date_entry_caceis"] = pd.to_datetime(panel["date_entry_caceis"], errors="coerce")
        panel["date_entry_poste"] = pd.to_datetime(panel["date_entry_poste"], errors="coerce")
        panel = panel.dropna(subset=["employee_id", "period"]).copy()

        panel = panel.drop_duplicates(subset=["employee_id", "period"], keep="first")

        panel["tenure_caceis_months"] = _months_diff(panel["date_entry_caceis"], panel["period"])
        panel["months_in_current_post"] = _months_diff(panel["date_entry_poste"], panel["period"])

        return panel.sort_values(["employee_id", "period"]).reset_index(drop=True)


    def detect_left_next_6m(employee_month: pd.DataFrame) -> pd.DataFrame:
        """
        KPI_PLAN step 2:
        Detect employees who disappear from the monthly panel for the following 6 months.
        """
        presence = employee_month[["employee_id", "period"]].drop_duplicates()
        ordered_months = sorted(presence["period"].unique())

        pivot = (
            presence.assign(present=1.0)
            .pivot(index="employee_id", columns="period", values="present")
            .reindex(columns=ordered_months)
            .fillna(0.0)
        )

        future_presence = 0.0
        for step in range(1, 7):
            future_presence = future_presence + pivot.shift(-step, axis=1).fillna(0.0)

        left_flag = (future_presence == 0.0).astype(float)
        if left_flag.shape[1] >= 6:
            left_flag.iloc[:, -6:] = float("nan")

        long = (
            left_flag.reset_index()
            .melt(id_vars="employee_id", var_name="period", value_name="left_next_6m")
            .sort_values(["employee_id", "period"])
            .reset_index(drop=True)
        )
        long["period"] = pd.to_datetime(long["period"], errors="coerce")
        return long


    def _read_absence_rows(file_path: Path) -> pd.DataFrame:
        required = {"Employee Code", "Date Absence", "Jours Ouvrés Absence"}
        xl = pd.ExcelFile(file_path)

        chosen_sheet: str | None = None
        chosen_header: int | None = None

        for sheet in xl.sheet_names:
            for header_row in range(0, 4):
                try:
                    preview = pd.read_excel(file_path, sheet_name=sheet, header=header_row, nrows=0)
                except Exception:
                    continue
                preview_cols = {str(col).strip() for col in preview.columns}
                if required.issubset(preview_cols):
                    chosen_sheet = sheet
                    chosen_header = header_row
                    break
            if chosen_sheet is not None:
                break

        if chosen_sheet is None or chosen_header is None:
            raise ValueError(f"Could not locate absence headers in file: {file_path}")

        df = pd.read_excel(
            file_path,
            sheet_name=chosen_sheet,
            header=chosen_header,
            usecols=["Employee Code", "Date Absence", "Jours Ouvrés Absence"],
        ).rename(
            columns={
                "Employee Code": "employee_id",
                "Date Absence": "absence_date",
                "Jours Ouvrés Absence": "absence_days",
            }
        )

        df["employee_id"] = _normalize_id(df["employee_id"])
        df["period"] = _to_month_start(df["absence_date"])
        df["absence_days"] = pd.to_numeric(df["absence_days"], errors="coerce").fillna(0.0)
        return df.dropna(subset=["employee_id", "period"])[["employee_id", "period", "absence_days"]]


    def load_absence_monthly(absence_paths: Iterable[Path]) -> pd.DataFrame:
        """
        KPI_PLAN step 3 (part):
        Aggregate absence by employee-month.
        """
        frames: list[pd.DataFrame] = []
        for path in absence_paths:
            if path.exists():
                frames.append(_read_absence_rows(path))

        if not frames:
            return pd.DataFrame(columns=["employee_id", "period", "absence_days"])

        all_absence = pd.concat(frames, ignore_index=True)
        return (
            all_absence.groupby(["employee_id", "period"], as_index=False)["absence_days"]
            .sum()
            .sort_values(["employee_id", "period"])
            .reset_index(drop=True)
        )


    def load_training_monthly(training_path: Path) -> pd.DataFrame:
        """
        KPI_PLAN step 3 (part):
        Aggregate training by employee-month.
        """
        raw = pd.read_excel(training_path, sheet_name="Final_CSV")

        if "Employee Code" not in raw.columns or "Total_Training_Hours" not in raw.columns:
            raise ValueError("Training source is missing required columns.")

        train = raw.rename(
            columns={
                "Employee Code": "employee_id",
                "Session_End_Date": "session_end_date",
                "Seesion_Start_Date": "session_start_date",
                "Total_Training_Hours": "training_hours",
                "Year": "year",
            }
        )

        train["employee_id"] = _normalize_id(train["employee_id"])
        train["training_hours"] = pd.to_numeric(train["training_hours"], errors="coerce").fillna(0.0)
        train["session_end_date"] = pd.to_datetime(train["session_end_date"], errors="coerce", dayfirst=True)
        train["session_start_date"] = pd.to_datetime(
            train["session_start_date"], errors="coerce", dayfirst=True
        )
        fallback_year = pd.to_numeric(train.get("year"), errors="coerce")
        fallback_date = pd.to_datetime(fallback_year, format="%Y", errors="coerce")
        event_date = train["session_end_date"].fillna(train["session_start_date"]).fillna(fallback_date)

        train["period"] = event_date.dt.to_period("M").dt.to_timestamp()
        train = train.dropna(subset=["employee_id", "period"])

        return (
            train.groupby(["employee_id", "period"], as_index=False)["training_hours"]
            .sum()
            .sort_values(["employee_id", "period"])
            .reset_index(drop=True)
        )


    def load_eae_scores(eae_path: Path) -> pd.DataFrame:
        """
        KPI_PLAN step 3 (part):
        Extract performance score from EAE data.
        """
        raw = pd.read_excel(eae_path, sheet_name="Database")
        raw.columns = [col.strip() if isinstance(col, str) else col for col in raw.columns]

        needed = ["IUG", "Année", "Note de performance"]
        missing = [col for col in needed if col not in raw.columns]
        if missing:
            raise ValueError(f"Missing columns in EAE source: {missing}")

        perf = raw.loc[:, needed].rename(
            columns={"IUG": "employee_id", "Année": "year", "Note de performance": "performance_score"}
        )

        perf["employee_id"] = _normalize_id(perf["employee_id"])
        perf["year"] = pd.to_numeric(perf["year"], errors="coerce")
        perf["performance_score"] = pd.to_numeric(perf["performance_score"], errors="coerce")
        perf = perf.dropna(subset=["employee_id", "year", "performance_score"])
        perf["year"] = perf["year"].astype(int)

        return (
            perf.groupby(["employee_id", "year"], as_index=False)["performance_score"]
            .mean()
            .sort_values(["employee_id", "year"])
            .reset_index(drop=True)
        )


    def _attach_trailing_12m(
        panel: pd.DataFrame, monthly_values: pd.DataFrame, input_col: str, output_col: str
    ) -> pd.DataFrame:
        if monthly_values.empty:
            with_default = panel.copy()
            with_default[output_col] = 0.0
            return with_default

        all_months = pd.date_range(panel["period"].min(), panel["period"].max(), freq="MS")
        all_ids = panel["employee_id"].dropna().unique()
        full_index = pd.MultiIndex.from_product([all_ids, all_months], names=["employee_id", "period"])

        series = monthly_values.set_index(["employee_id", "period"])[input_col]
        dense = series.reindex(full_index, fill_value=0.0).reset_index(name=input_col)
        dense[output_col] = dense.groupby("employee_id")[input_col].transform(
            lambda s: s.rolling(window=12, min_periods=1).sum()
        )

        return panel.merge(dense[["employee_id", "period", output_col]], on=["employee_id", "period"], how="left")


    def enrich_kpi2_features(
        employee_month: pd.DataFrame,
        left_next_6m: pd.DataFrame,
        eae_scores: pd.DataFrame,
        training_monthly: pd.DataFrame,
        absence_monthly: pd.DataFrame,
    ) -> pd.DataFrame:
        """
        KPI_PLAN step 3:
        Join EAE, training, and absence features to employee-month data.
        """
        base = employee_month.merge(left_next_6m, on=["employee_id", "period"], how="left")
        base = _attach_trailing_12m(base, absence_monthly, "absence_days", "absence_days_12m")
        base = _attach_trailing_12m(base, training_monthly, "training_hours", "training_hours_12m")

        base["year"] = base["period"].dt.year
        joined = base.merge(eae_scores, on=["employee_id", "year"], how="left")

        emp_perf_avg = eae_scores.groupby("employee_id")["performance_score"].mean()
        global_perf_median = eae_scores["performance_score"].median()
        joined["performance_score"] = joined["performance_score"].fillna(joined["employee_id"].map(emp_perf_avg))
        joined["performance_score"] = joined["performance_score"].fillna(global_perf_median)

        joined["absence_days_12m"] = joined["absence_days_12m"].fillna(0.0)
        joined["training_hours_12m"] = joined["training_hours_12m"].fillna(0.0)
        joined["tenure_caceis_months"] = joined["tenure_caceis_months"].fillna(0.0)
        joined["months_in_current_post"] = joined["months_in_current_post"].fillna(0.0)

        return joined


    def compute_risk_scores(kpi2_features: pd.DataFrame) -> pd.DataFrame:
        """
        KPI_PLAN step 4:
        Create weighted Workforce Value-at-Risk score.
        """
        df = kpi2_features.copy()

        df["new_joiner_risk"] = ((24.0 - df["tenure_caceis_months"]).clip(lower=0.0, upper=24.0)) / 24.0
        df["low_performance_risk"] = ((5.0 - df["performance_score"]).clip(lower=0.0, upper=4.0)) / 4.0

        absence_p90 = df["absence_days_12m"].quantile(0.90)
        if pd.isna(absence_p90) or absence_p90 <= 0:
            absence_p90 = 1.0
        df["high_absence_risk"] = (df["absence_days_12m"] / absence_p90).clip(lower=0.0, upper=1.0)

        training_p75 = df["training_hours_12m"].quantile(0.75)
        if pd.isna(training_p75) or training_p75 <= 0:
            training_p75 = 1.0
        df["low_training_risk"] = 1.0 - (df["training_hours_12m"] / training_p75).clip(lower=0.0, upper=1.0)

        df["long_time_in_post_risk"] = (
            (df["months_in_current_post"] - 36.0) / 60.0
        ).clip(lower=0.0, upper=1.0)

        segment_cols = ["country", "entity", "job"]
        labeled = df.dropna(subset=["left_next_6m"]).copy()
        global_leave_rate = labeled["left_next_6m"].mean() if not labeled.empty else 0.0
        smoothing = 20.0

        if not labeled.empty:
            segment_stats = (
                labeled.groupby(segment_cols)["left_next_6m"]
                .agg(["sum", "count"])
                .reset_index()
                .rename(columns={"sum": "leavers", "count": "observations"})
            )
            segment_stats["segment_risk"] = (
                segment_stats["leavers"] + smoothing * global_leave_rate
            ) / (segment_stats["observations"] + smoothing)
            df = df.merge(segment_stats[segment_cols + ["segment_risk"]], on=segment_cols, how="left")
        else:
            df["segment_risk"] = global_leave_rate

        df["segment_risk"] = df["segment_risk"].fillna(global_leave_rate).clip(lower=0.0, upper=1.0)

        df["risk_score"] = (
            0.25 * df["new_joiner_risk"]
            + 0.20 * df["low_performance_risk"]
            + 0.20 * df["high_absence_risk"]
            + 0.15 * df["low_training_risk"]
            + 0.10 * df["long_time_in_post_risk"]
            + 0.10 * df["segment_risk"]
        ).clip(lower=0.0, upper=1.0)

        q_low = float(df["risk_score"].quantile(0.33))
        q_high = float(df["risk_score"].quantile(0.67))
        if q_high <= q_low:
            q_low, q_high = 0.33, 0.66

        df["risk_band"] = pd.cut(
            df["risk_score"],
            bins=[-0.001, q_low, q_high, 1.0],
            labels=["Low", "Medium", "High"],
            include_lowest=True,
        )

        return df.sort_values(["period", "risk_score"], ascending=[True, False]).reset_index(drop=True)


    def build_kpi2_result(config: KPI2Config) -> KPI2Result:
        """
        KPI_PLAN steps 1-5 end-to-end builder.
        """
        panel = build_employee_month_table(config.panel_path)
        left_next_6m = detect_left_next_6m(panel)
        absence_monthly = load_absence_monthly(config.absence_paths)
        training_monthly = load_training_monthly(config.training_path)
        eae_scores = load_eae_scores(config.eae_path)

        features = enrich_kpi2_features(
            employee_month=panel,
            left_next_6m=left_next_6m,
            eae_scores=eae_scores,
            training_monthly=training_monthly,
            absence_monthly=absence_monthly,
        )
        scored = compute_risk_scores(features)

        latest_period = scored["period"].max()
        latest = scored[scored["period"] == latest_period].copy()

        latest_risk_table = latest.sort_values("risk_score", ascending=False).reset_index(drop=True)

        entity_risk = (
            latest.groupby("entity", as_index=False)
            .agg(
                employees=("employee_id", "nunique"),
                avg_risk_score=("risk_score", "mean"),
                high_risk_share=("risk_band", lambda s: float((s == "High").mean())),
            )
            .sort_values(["avg_risk_score", "employees"], ascending=[False, False])
            .reset_index(drop=True)
        )

        job_risk = (
            latest.groupby("job", as_index=False)
            .agg(
                employees=("employee_id", "nunique"),
                avg_risk_score=("risk_score", "mean"),
                high_risk_share=("risk_band", lambda s: float((s == "High").mean())),
            )
            .sort_values(["avg_risk_score", "employees"], ascending=[False, False])
            .reset_index(drop=True)
        )

        segment_risk = (
            latest.groupby(["country", "entity", "job"], as_index=False)
            .agg(
                employees=("employee_id", "nunique"),
                avg_risk_score=("risk_score", "mean"),
                high_risk_share=("risk_band", lambda s: float((s == "High").mean())),
            )
            .sort_values(["avg_risk_score", "employees"], ascending=[False, False])
            .reset_index(drop=True)
        )

        risk_trend = (
            scored.groupby("period", as_index=False)
            .agg(
                avg_risk_score=("risk_score", "mean"),
                high_risk_share=("risk_band", lambda s: float((s == "High").mean())),
                observed_leave_rate=("left_next_6m", "mean"),
                employees=("employee_id", "nunique"),
            )
            .sort_values("period")
            .reset_index(drop=True)
        )

        return KPI2Result(
            risk_table=scored,
            latest_risk_table=latest_risk_table,
            entity_risk=entity_risk,
            job_risk=job_risk,
            segment_risk=segment_risk,
            risk_trend=risk_trend,
        )


    def summarize_kpi2(result: KPI2Result) -> list[str]:
        """
        KPI_PLAN step 5:
        Provide short business interpretation for top-risk segments.
        """
        latest = result.latest_risk_table
        trend = result.risk_trend

        if latest.empty:
            return ["No risk records were generated."]

        latest_period = latest["period"].max()
        high_share = float((latest["risk_band"] == "High").mean())
        avg_risk = float(latest["risk_score"].mean())
        top_entity = result.entity_risk.iloc[0]["entity"] if not result.entity_risk.empty else "N/A"
        top_job = result.job_risk.iloc[0]["job"] if not result.job_risk.empty else "N/A"

        messages = [
            (
                f"Latest month in panel: {latest_period:%Y-%m}. "
                f"Average risk score is {avg_risk:.3f} with {high_share:.1%} employees in High risk."
            ),
            f"Highest-risk entity (by average score): {top_entity}.",
            f"Highest-risk job family (by average score): {top_job}.",
        ]

        if not trend.empty and trend["avg_risk_score"].notna().sum() >= 2:
            start = trend.iloc[0]
            end = trend.iloc[-1]
            delta = float(end["avg_risk_score"] - start["avg_risk_score"])
            messages.append(
                f"Average risk score trend moved from {start['avg_risk_score']:.3f} "
                f"to {end['avg_risk_score']:.3f} ({delta:+.3f})."
            )

        return messages

    return KPI2Config, KPI2Result, build_kpi2_result, summarize_kpi2

KPI2Config, KPI2Result, build_kpi2_result, summarize_kpi2 = _register_kpi2_pipeline()


In [26]:
# KPI 3 Pipeline Code 
def _register_kpi3_pipeline():

    import re
    import unicodedata
    from dataclasses import dataclass
    from pathlib import Path

    import pandas as pd


    DEFAULT_PANEL_PATH = Path("Sujet Alberthon/HR Data/Data.xlsx")
    DEFAULT_EAE_PATH = Path(
        "Sujet Alberthon/HR Data/20250218 - Stats CACEIS EAE EP 18-02-2025 Version Définitive cloture.xlsx"
    )
    DEFAULT_TRAINING_PATH = Path("Sujet Alberthon/Training/Training_Records_Unnamed.xlsx")
    DEFAULT_QUICK_REVIEW_PATH = Path("Sujet Alberthon/Training/Quick_Review_Unnamed.xlsx")
    DEFAULT_COLD_REVIEW_PATH = Path("Sujet Alberthon/Training/Cold_Review_Unnamed.xlsx")


    @dataclass(frozen=True)
    class KPI3Config:
        panel_path: Path = DEFAULT_PANEL_PATH
        eae_path: Path = DEFAULT_EAE_PATH
        training_path: Path = DEFAULT_TRAINING_PATH
        quick_review_path: Path = DEFAULT_QUICK_REVIEW_PATH
        cold_review_path: Path = DEFAULT_COLD_REVIEW_PATH


    @dataclass
    class KPI3Result:
        employee_year_table: pd.DataFrame
        latest_year_table: pd.DataFrame
        yearly_summary: pd.DataFrame
        trained_vs_non_trained: pd.DataFrame
        entity_learning: pd.DataFrame
        scatter_training_performance: pd.DataFrame
        scatter_training_impact: pd.DataFrame


    def _normalize_id(values: pd.Series) -> pd.Series:
        cleaned = values.astype(str).str.strip()
        return cleaned.replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA, "None": pd.NA, "NaT": pd.NA})


    def _normalize_text(value: object) -> str:
        if value is None:
            return ""
        if isinstance(value, float) and pd.isna(value):
            return ""
        text = str(value).strip().replace("\xa0", " ")
        text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
        text = text.lower()
        text = re.sub(r"[^a-z0-9]+", " ", text)
        return " ".join(text.split())


    def _extract_event_year(
        frame: pd.DataFrame, date_columns: list[str], fallback_year_column: str | None = None
    ) -> pd.Series:
        event_date = pd.Series(pd.NaT, index=frame.index, dtype="datetime64[ns]")
        for col in date_columns:
            if col in frame.columns:
                parsed = pd.to_datetime(frame[col], errors="coerce", dayfirst=True)
                event_date = event_date.fillna(parsed)

        if fallback_year_column and fallback_year_column in frame.columns:
            year_values = pd.to_numeric(frame[fallback_year_column], errors="coerce")
            fallback_date = pd.to_datetime(year_values, format="%Y", errors="coerce")
            event_date = event_date.fillna(fallback_date)

        return event_date.dt.year


    def build_employee_year_base(panel_path: Path) -> pd.DataFrame:
        """
        KPI_PLAN step 3 foundation:
        Employee-year population used to compare trained vs non-trained employees.
        """
        raw = pd.read_excel(panel_path, sheet_name="Sheet1")
        use_cols = ["PERIOD", "ID Employee", "COUNTRY_GROUP_LABEL_EN", "ENTITY_LABEL_LOCAL", "POSTE_LABEL_LOCAL"]
        missing_cols = [col for col in use_cols if col not in raw.columns]
        if missing_cols:
            raise ValueError(f"Missing columns in panel source: {missing_cols}")

        panel = raw.loc[:, use_cols].rename(
            columns={
                "PERIOD": "period",
                "ID Employee": "employee_id",
                "COUNTRY_GROUP_LABEL_EN": "country",
                "ENTITY_LABEL_LOCAL": "entity",
                "POSTE_LABEL_LOCAL": "job",
            }
        )
        panel["employee_id"] = _normalize_id(panel["employee_id"])
        panel["period"] = pd.to_datetime(panel["period"], errors="coerce", dayfirst=True)
        panel["year"] = panel["period"].dt.year
        panel = panel.dropna(subset=["employee_id", "period", "year"]).copy()
        panel["year"] = panel["year"].astype(int)

        # Keep the last monthly row in each employee-year to retain a stable segment label.
        ordered = panel.sort_values(["employee_id", "year", "period"])
        base = (
            ordered.groupby(["employee_id", "year"], as_index=False)
            .tail(1)
            .loc[:, ["employee_id", "year", "country", "entity", "job"]]
            .reset_index(drop=True)
        )
        return base


    def aggregate_training_records(training_path: Path) -> pd.DataFrame:
        """
        KPI_PLAN step 1:
        Aggregate training records by employee-year.
        """
        raw = pd.read_excel(training_path, sheet_name="Final_CSV")
        needed = ["Employee Code", "Status", "Total_Training_Hours", "Session_ID", "Certifications"]
        missing = [col for col in needed if col not in raw.columns]
        if missing:
            raise ValueError(f"Missing columns in training source: {missing}")

        train = raw.rename(
            columns={
                "Employee Code": "employee_id",
                "Status": "status",
                "Total_Training_Hours": "training_hours",
                "Session_ID": "session_id",
                "Certifications": "certifications",
            }
        )
        train["employee_id"] = _normalize_id(train["employee_id"])
        train["training_hours"] = pd.to_numeric(train["training_hours"], errors="coerce").fillna(0.0)
        train["year"] = _extract_event_year(
            train,
            date_columns=["Session_End_Date", "Seesion_Start_Date"],
            fallback_year_column="Year",
        )
        train["status_norm"] = train["status"].map(_normalize_text)

        effective_status = {"realise", "completed", "completee"}
        train = train[
            train["status_norm"].isin(effective_status)
            | ((train["training_hours"] > 0.0) & train["status_norm"].eq(""))
        ].copy()

        train = train.dropna(subset=["employee_id", "year"]).copy()
        train["year"] = train["year"].astype(int)
        train["certification_flag"] = train["certifications"].map(
            lambda v: 1.0 if _normalize_text(v).startswith("yes") else 0.0
        )
        train["session_non_null"] = train["session_id"].notna().astype(float)

        grouped = (
            train.groupby(["employee_id", "year"], as_index=False)
            .agg(
                training_hours=("training_hours", "sum"),
                training_count=("session_non_null", "sum"),
                certification_flag=("certification_flag", "max"),
            )
            .sort_values(["employee_id", "year"])
            .reset_index(drop=True)
        )
        return grouped


    def aggregate_quick_reviews(quick_review_path: Path) -> pd.DataFrame:
        """
        KPI_PLAN step 2 (part):
        Aggregate quick review scores by employee-year.
        """
        raw = pd.read_excel(quick_review_path, sheet_name="Data")
        needed = ["Matricule", "Note générale"]
        missing = [col for col in needed if col not in raw.columns]
        if missing:
            raise ValueError(f"Missing columns in quick-review source: {missing}")

        quick = raw.rename(columns={"Matricule": "employee_id", "Note générale": "quick_review_score"})
        quick["employee_id"] = _normalize_id(quick["employee_id"])
        quick["year"] = _extract_event_year(
            quick,
            date_columns=["Date de fin de session", "Date de début de session", "Date"],
        )
        quick["quick_review_score"] = pd.to_numeric(quick["quick_review_score"], errors="coerce")
        quick["session_non_null"] = quick.get("ID de session", pd.Series(index=quick.index)).notna().astype(float)
        quick = quick.dropna(subset=["employee_id", "year", "quick_review_score"]).copy()
        quick["year"] = quick["year"].astype(int)

        grouped = (
            quick.groupby(["employee_id", "year"], as_index=False)
            .agg(
                quick_review_score=("quick_review_score", "mean"),
                quick_review_count=("session_non_null", "sum"),
            )
            .sort_values(["employee_id", "year"])
            .reset_index(drop=True)
        )
        return grouped


    def _map_cold_answer(value: object) -> float:
        normalized = _normalize_text(value)
        if normalized == "":
            return float("nan")
        if normalized == "oui":
            return 1.0
        if normalized.startswith("oui tout a fait"):
            return 1.0
        if normalized.startswith("oui en partie"):
            return 0.5
        if normalized.startswith("non"):
            return 0.0
        return float("nan")


    def _select_cold_answer_columns(columns: list[object]) -> list[str]:
        patterns = [
            "considerez vous que cette formation vous a permis",
            "la formation a t elle repondu a vos attentes initiales",
            "estimez vous que la formation etait en adequation",
            "recommanderiez vous ce stage",
            "utilisez vous les connaissances acquises",
        ]
        selected: list[str] = []
        for col in columns:
            col_name = str(col)
            normalized = _normalize_text(col_name)
            if any(pattern in normalized for pattern in patterns):
                selected.append(col_name)
        return selected


    def aggregate_cold_reviews(cold_review_path: Path) -> pd.DataFrame:
        """
        KPI_PLAN step 2 (part):
        Aggregate converted cold-review impact score by employee-year.
        """
        raw = pd.read_excel(cold_review_path, sheet_name="Data")
        if "Matricule" not in raw.columns:
            raise ValueError("Missing 'Matricule' column in cold-review source.")

        answer_columns = _select_cold_answer_columns(list(raw.columns))
        if not answer_columns:
            raise ValueError("No cold-review impact answer columns detected.")

        cold = raw.rename(columns={"Matricule": "employee_id"})
        cold["employee_id"] = _normalize_id(cold["employee_id"])
        cold["year"] = _extract_event_year(
            cold,
            date_columns=["Date de fin de session", "Date de début de session", "Date"],
        )
        cold["status_norm"] = cold.get("Status", pd.Series(index=cold.index)).map(_normalize_text)

        mapped = cold.loc[:, answer_columns].apply(lambda col: col.map(_map_cold_answer))
        cold["cold_impact_score"] = mapped.mean(axis=1, skipna=True)

        blocked_status = {"annulee", "non envoyee", "en attente"}
        valid_rows = ~cold["status_norm"].isin(blocked_status) | cold["status_norm"].eq("")

        cold = cold[valid_rows].dropna(subset=["employee_id", "year", "cold_impact_score"]).copy()
        cold["year"] = cold["year"].astype(int)

        grouped = (
            cold.groupby(["employee_id", "year"], as_index=False)
            .agg(
                cold_impact_score=("cold_impact_score", "mean"),
                cold_review_count=("cold_impact_score", "count"),
            )
            .sort_values(["employee_id", "year"])
            .reset_index(drop=True)
        )
        return grouped


    def load_eae_scores(eae_path: Path) -> pd.DataFrame:
        """
        KPI_PLAN step 3 (part):
        Join annual performance score from EAE.
        """
        raw = pd.read_excel(eae_path, sheet_name="Database")
        raw.columns = [col.strip() if isinstance(col, str) else col for col in raw.columns]

        needed = ["IUG", "Année", "Note de performance"]
        missing = [col for col in needed if col not in raw.columns]
        if missing:
            raise ValueError(f"Missing columns in EAE source: {missing}")

        perf = raw.loc[:, needed].rename(
            columns={"IUG": "employee_id", "Année": "year", "Note de performance": "performance_score"}
        )
        perf["employee_id"] = _normalize_id(perf["employee_id"])
        perf["year"] = pd.to_numeric(perf["year"], errors="coerce")
        perf["performance_score"] = pd.to_numeric(perf["performance_score"], errors="coerce")
        perf = perf.dropna(subset=["employee_id", "year", "performance_score"]).copy()
        perf["year"] = perf["year"].astype(int)

        return (
            perf.groupby(["employee_id", "year"], as_index=False)["performance_score"]
            .mean()
            .sort_values(["employee_id", "year"])
            .reset_index(drop=True)
        )


    def build_learning_performance_table(
        employee_year_base: pd.DataFrame,
        training_agg: pd.DataFrame,
        quick_agg: pd.DataFrame,
        cold_agg: pd.DataFrame,
        eae_scores: pd.DataFrame,
    ) -> pd.DataFrame:
        """
        KPI_PLAN step 3:
        Join training, review scores, and EAE performance.
        """
        table = employee_year_base.merge(training_agg, on=["employee_id", "year"], how="left")
        table = table.merge(quick_agg, on=["employee_id", "year"], how="left")
        table = table.merge(cold_agg, on=["employee_id", "year"], how="left")
        table = table.merge(eae_scores, on=["employee_id", "year"], how="left")

        for col in ["training_hours", "training_count", "quick_review_count", "cold_review_count"]:
            if col in table.columns:
                table[col] = pd.to_numeric(table[col], errors="coerce").fillna(0.0)

        table["certification_flag"] = pd.to_numeric(table.get("certification_flag"), errors="coerce").fillna(0.0)
        table["certification_flag"] = table["certification_flag"].clip(lower=0.0, upper=1.0)

        table["quick_review_score"] = pd.to_numeric(table.get("quick_review_score"), errors="coerce")
        table["cold_impact_score"] = pd.to_numeric(table.get("cold_impact_score"), errors="coerce")

        table["trained_flag"] = ((table["training_hours"] > 0.0) | (table["training_count"] > 0.0)).astype(int)

        positive_hours_p95 = (
            table.loc[table["training_hours"] > 0.0]
            .groupby("year")["training_hours"]
            .quantile(0.95)
            .rename("p95_hours")
        )
        training_denominator = table["year"].map(positive_hours_p95).fillna(1.0)
        training_denominator = training_denominator.where(training_denominator > 0.0, 1.0)

        table["training_hours_norm"] = (table["training_hours"] / training_denominator).clip(lower=0.0, upper=1.0)
        table["quick_review_norm"] = ((table["quick_review_score"] - 1.0) / 4.0).clip(lower=0.0, upper=1.0)
        table["cold_impact_norm"] = table["cold_impact_score"].clip(lower=0.0, upper=1.0)

        table["training_hours_norm"] = table["training_hours_norm"].fillna(0.0)
        table["quick_review_norm"] = table["quick_review_norm"].fillna(0.0)
        table["cold_impact_norm"] = table["cold_impact_norm"].fillna(0.0)

        table["learning_to_performance_index"] = (
            0.40 * table["training_hours_norm"]
            + 0.30 * table["cold_impact_norm"]
            + 0.20 * table["quick_review_norm"]
            + 0.10 * table["certification_flag"]
        ).clip(lower=0.0, upper=1.0)

        return table.sort_values(["year", "learning_to_performance_index"], ascending=[True, False]).reset_index(
            drop=True
        )


    def build_kpi3_result(config: KPI3Config) -> KPI3Result:
        """
        KPI_PLAN steps 1-5 end-to-end builder.
        """
        base = build_employee_year_base(config.panel_path)
        training_agg = aggregate_training_records(config.training_path)
        quick_agg = aggregate_quick_reviews(config.quick_review_path)
        cold_agg = aggregate_cold_reviews(config.cold_review_path)
        eae_scores = load_eae_scores(config.eae_path)

        table = build_learning_performance_table(
            employee_year_base=base,
            training_agg=training_agg,
            quick_agg=quick_agg,
            cold_agg=cold_agg,
            eae_scores=eae_scores,
        )

        latest_year = int(table["year"].max()) if not table.empty else None
        latest_year_table = (
            table.loc[table["year"] == latest_year].sort_values("learning_to_performance_index", ascending=False)
            if latest_year is not None
            else table.copy()
        ).reset_index(drop=True)

        yearly_summary = (
            table.groupby("year", as_index=False)
            .agg(
                employees=("employee_id", "nunique"),
                trained_share=("trained_flag", "mean"),
                certified_share=("certification_flag", "mean"),
                avg_training_hours=("training_hours", "mean"),
                avg_training_count=("training_count", "mean"),
                avg_quick_review_score=("quick_review_score", "mean"),
                avg_cold_impact_score=("cold_impact_score", "mean"),
                avg_learning_index=("learning_to_performance_index", "mean"),
                avg_performance_score=("performance_score", "mean"),
                performance_coverage=("performance_score", lambda s: float(s.notna().mean())),
            )
            .sort_values("year")
            .reset_index(drop=True)
        )

        status_summary = (
            table.groupby(["year", "trained_flag"], as_index=False)
            .agg(
                employees=("employee_id", "nunique"),
                avg_performance_score=("performance_score", "mean"),
                avg_learning_index=("learning_to_performance_index", "mean"),
                avg_training_hours=("training_hours", "mean"),
                avg_cold_impact_score=("cold_impact_score", "mean"),
            )
            .sort_values(["year", "trained_flag"])
            .reset_index(drop=True)
        )

        trained = status_summary[status_summary["trained_flag"] == 1].rename(
            columns={
                "employees": "trained_employees",
                "avg_performance_score": "trained_avg_performance",
                "avg_learning_index": "trained_avg_learning_index",
                "avg_training_hours": "trained_avg_training_hours",
                "avg_cold_impact_score": "trained_avg_cold_impact_score",
            }
        )
        non_trained = status_summary[status_summary["trained_flag"] == 0].rename(
            columns={
                "employees": "non_trained_employees",
                "avg_performance_score": "non_trained_avg_performance",
                "avg_learning_index": "non_trained_avg_learning_index",
                "avg_training_hours": "non_trained_avg_training_hours",
                "avg_cold_impact_score": "non_trained_avg_cold_impact_score",
            }
        )

        trained_vs_non_trained = trained.merge(non_trained, on="year", how="outer", suffixes=("", "_drop"))
        trained_vs_non_trained = trained_vs_non_trained.loc[
            :,
            [
                "year",
                "trained_employees",
                "non_trained_employees",
                "trained_avg_performance",
                "non_trained_avg_performance",
                "trained_avg_learning_index",
                "non_trained_avg_learning_index",
                "trained_avg_training_hours",
                "non_trained_avg_training_hours",
                "trained_avg_cold_impact_score",
                "non_trained_avg_cold_impact_score",
            ],
        ].sort_values("year")
        trained_vs_non_trained["performance_gap_trained_minus_non_trained"] = (
            trained_vs_non_trained["trained_avg_performance"]
            - trained_vs_non_trained["non_trained_avg_performance"]
        )
        trained_vs_non_trained["learning_index_gap_trained_minus_non_trained"] = (
            trained_vs_non_trained["trained_avg_learning_index"]
            - trained_vs_non_trained["non_trained_avg_learning_index"]
        )
        trained_vs_non_trained = trained_vs_non_trained.reset_index(drop=True)

        entity_learning = (
            latest_year_table.groupby("entity", as_index=False)
            .agg(
                employees=("employee_id", "nunique"),
                trained_share=("trained_flag", "mean"),
                avg_learning_index=("learning_to_performance_index", "mean"),
                avg_training_hours=("training_hours", "mean"),
                avg_performance_score=("performance_score", "mean"),
            )
            .sort_values(["avg_learning_index", "employees"], ascending=[False, False])
            .reset_index(drop=True)
        )

        scatter_training_performance = (
            table.loc[(table["training_hours"] > 0.0) & table["performance_score"].notna()]
            .loc[
                :,
                [
                    "employee_id",
                    "year",
                    "country",
                    "entity",
                    "job",
                    "training_hours",
                    "training_count",
                    "performance_score",
                    "learning_to_performance_index",
                ],
            ]
            .reset_index(drop=True)
        )

        scatter_training_impact = (
            table.loc[(table["training_hours"] > 0.0) & table["cold_impact_score"].notna()]
            .loc[
                :,
                [
                    "employee_id",
                    "year",
                    "country",
                    "entity",
                    "job",
                    "training_hours",
                    "training_count",
                    "cold_impact_score",
                    "learning_to_performance_index",
                ],
            ]
            .reset_index(drop=True)
        )

        return KPI3Result(
            employee_year_table=table,
            latest_year_table=latest_year_table,
            yearly_summary=yearly_summary,
            trained_vs_non_trained=trained_vs_non_trained,
            entity_learning=entity_learning,
            scatter_training_performance=scatter_training_performance,
            scatter_training_impact=scatter_training_impact,
        )


    def summarize_kpi3(result: KPI3Result) -> list[str]:
        """
        KPI_PLAN steps 4-5:
        Short business interpretation for learning and performance signal.
        """
        table = result.employee_year_table
        if table.empty:
            return ["No KPI 3 records were generated."]

        latest_year = int(table["year"].max())
        latest = table[table["year"] == latest_year]
        avg_index = float(latest["learning_to_performance_index"].mean())
        trained_share = float(latest["trained_flag"].mean())
        lines = [
            (
                f"Latest year: {latest_year}. Average Learning-to-Performance Index is {avg_index:.3f}, "
                f"with {trained_share:.1%} employees trained."
            )
        ]

        compare_valid = result.trained_vs_non_trained.dropna(
            subset=["performance_gap_trained_minus_non_trained"]
        ).sort_values("year")
        if not compare_valid.empty:
            row = compare_valid.iloc[-1]
            lines.append(
                "Performance gap (trained - non-trained) in latest year with EAE coverage "
                f"({int(row['year'])}): {row['performance_gap_trained_minus_non_trained']:+.3f} points."
            )

        if not result.scatter_training_performance.empty:
            corr_perf = result.scatter_training_performance["training_hours"].corr(
                result.scatter_training_performance["performance_score"]
            )
            if pd.notna(corr_perf):
                lines.append(f"Correlation between training hours and performance score: {corr_perf:+.3f}.")

        if not result.scatter_training_impact.empty:
            corr_impact = result.scatter_training_impact["training_hours"].corr(
                result.scatter_training_impact["cold_impact_score"]
            )
            if pd.notna(corr_impact):
                lines.append(f"Correlation between training hours and cold impact score: {corr_impact:+.3f}.")

        if not result.entity_learning.empty:
            top_entity = result.entity_learning.iloc[0]["entity"]
            lines.append(f"Highest learning index entity in latest year: {top_entity}.")

        return lines

    return KPI3Config, KPI3Result, build_kpi3_result, summarize_kpi3

KPI3Config, KPI3Result, build_kpi3_result, summarize_kpi3 = _register_kpi3_pipeline()


In [27]:
# KPI 4 Pipeline Code
def _register_kpi4_pipeline():

    from dataclasses import dataclass
    from pathlib import Path
    from typing import Iterable, Literal

    import pandas as pd

    Scope = Literal["group", "europe"]



    DEFAULT_PANEL_PATH = Path("Sujet Alberthon/HR Data/Data.xlsx")
    DEFAULT_FINANCE_PATH = Path("Sujet Alberthon/Finance/AlbertSchool_CACEIS_PL-FTE_22-25_Sent.xlsx")
    DEFAULT_ABSENCE_PATHS = (
        Path("Sujet Alberthon/HR Data/Absentéisme_-_détail_affectation_-_Bilan_social.xlsx"),
        Path("Sujet Alberthon/HR Data/Absentéisme_-_détail_affectation_-_Bilan_social (1).xlsx"),
        Path(
            "Sujet Alberthon/HR Data/20260121 - Absentéisme_-_détail_affectation_-_Bilan_social 2025.xlsx"
        ),
    )


    @dataclass(frozen=True)
    class KPI4Config:
        panel_path: Path = DEFAULT_PANEL_PATH
        finance_path: Path = DEFAULT_FINANCE_PATH
        finance_scope: Scope = "group"
        absence_paths: tuple[Path, ...] = DEFAULT_ABSENCE_PATHS


    @dataclass
    class KPI4Result:
        absence_standardized_rows: pd.DataFrame
        employee_month_entity_reason: pd.DataFrame
        monthly_drag: pd.DataFrame
        yearly_drag: pd.DataFrame
        entity_year_drag: pd.DataFrame
        reason_group_year_drag: pd.DataFrame
        reason_detail_year_drag: pd.DataFrame
        latest_year_entity_drag: pd.DataFrame
        latest_year_reason_group_drag: pd.DataFrame
        latest_year_reason_detail_drag: pd.DataFrame


    def _normalize_id(values: pd.Series) -> pd.Series:
        cleaned = values.astype(str).str.strip()
        return cleaned.replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA, "None": pd.NA, "NaT": pd.NA})


    def _to_month_start(values: pd.Series) -> pd.Series:
        parsed = pd.to_datetime(values, errors="coerce", dayfirst=True)
        return parsed.dt.to_period("M").dt.to_timestamp()


    def _safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
        denominator_num = pd.to_numeric(denominator, errors="coerce")
        return numerator / denominator_num.where(denominator_num > 0.0)


    def _read_absence_rows(file_path: Path) -> pd.DataFrame:
        required = {"Employee Code", "Date Absence", "Jours Ouvrés Absence"}
        optional = [
            "Motif Jour Absence",
            "Regroupement Jour Absences",
            "Société",
        ]

        xl = pd.ExcelFile(file_path)
        chosen_sheet: str | None = None
        chosen_header: int | None = None
        chosen_columns: list[str] = []

        for sheet in xl.sheet_names:
            for header_row in range(0, 4):
                try:
                    preview = pd.read_excel(file_path, sheet_name=sheet, header=header_row, nrows=0)
                except Exception:
                    continue

                columns = [str(col).strip() for col in preview.columns]
                if required.issubset(set(columns)):
                    chosen_sheet = sheet
                    chosen_header = header_row
                    chosen_columns = columns
                    break

            if chosen_sheet is not None:
                break

        if chosen_sheet is None or chosen_header is None:
            raise ValueError(f"Could not locate expected absence headers in file: {file_path}")

        usecols = [col for col in [*required, *optional] if col in chosen_columns]
        raw = pd.read_excel(file_path, sheet_name=chosen_sheet, header=chosen_header, usecols=usecols)

        absence = raw.rename(
            columns={
                "Employee Code": "employee_id",
                "Date Absence": "absence_date",
                "Jours Ouvrés Absence": "absence_days",
                "Motif Jour Absence": "reason_detail",
                "Regroupement Jour Absences": "reason_group",
                "Société": "absence_entity_raw",
            }
        )

        absence["employee_id"] = _normalize_id(absence["employee_id"])
        absence["period"] = _to_month_start(absence["absence_date"])
        absence["year"] = absence["period"].dt.year
        absence["absence_days"] = pd.to_numeric(absence["absence_days"], errors="coerce").fillna(0.0)
        absence["absence_days"] = absence["absence_days"].clip(lower=0.0)
        absence["reason_detail"] = (
            absence.get("reason_detail", pd.Series(index=absence.index))
            .astype(str)
            .str.strip()
            .replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA})
        )
        absence["reason_group"] = (
            absence.get("reason_group", pd.Series(index=absence.index))
            .astype(str)
            .str.strip()
            .replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA})
        )
        absence["absence_entity_raw"] = (
            absence.get("absence_entity_raw", pd.Series(index=absence.index))
            .astype(str)
            .str.strip()
            .replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA})
        )
        absence["reason_group"] = absence["reason_group"].fillna(absence["reason_detail"]).fillna("Unknown")
        absence["reason_detail"] = absence["reason_detail"].fillna("Unknown")

        absence = absence.dropna(subset=["employee_id", "period", "year"]).copy()
        absence["year"] = absence["year"].astype(int)

        return absence.loc[
            :,
            [
                "employee_id",
                "period",
                "year",
                "absence_entity_raw",
                "reason_group",
                "reason_detail",
                "absence_days",
            ],
        ]


    def load_absence_standardized(absence_paths: Iterable[Path]) -> pd.DataFrame:
        """
        KPI_PLAN step 1-2:
        Load absence files and standardize employee/month/reason/day fields.
        """
        frames: list[pd.DataFrame] = []
        for path in absence_paths:
            if path.exists():
                frames.append(_read_absence_rows(path))

        if not frames:
            return pd.DataFrame(
                columns=[
                    "employee_id",
                    "period",
                    "year",
                    "absence_entity_raw",
                    "reason_group",
                    "reason_detail",
                    "absence_days",
                ]
            )

        stacked = pd.concat(frames, ignore_index=True)
        return (
            stacked.groupby(
                ["employee_id", "period", "year", "absence_entity_raw", "reason_group", "reason_detail"],
                as_index=False,
            )["absence_days"]
            .sum()
            .sort_values(["period", "employee_id"])
            .reset_index(drop=True)
        )


    def load_employee_month_dimension(panel_path: Path) -> pd.DataFrame:
        """
        Panel dimensions used for employee count denominator and entity enrichment.
        """
        raw = pd.read_excel(panel_path, sheet_name="Sheet1")
        use_cols = ["PERIOD", "ID Employee", "COUNTRY_GROUP_LABEL_EN", "ENTITY_LABEL_LOCAL", "POSTE_LABEL_LOCAL"]
        missing_cols = [col for col in use_cols if col not in raw.columns]
        if missing_cols:
            raise ValueError(f"Missing columns in panel source: {missing_cols}")

        panel = raw.loc[:, use_cols].rename(
            columns={
                "PERIOD": "period",
                "ID Employee": "employee_id",
                "COUNTRY_GROUP_LABEL_EN": "country",
                "ENTITY_LABEL_LOCAL": "panel_entity",
                "POSTE_LABEL_LOCAL": "job",
            }
        )
        panel["employee_id"] = _normalize_id(panel["employee_id"])
        panel["period"] = _to_month_start(panel["period"])
        panel = panel.dropna(subset=["employee_id", "period"]).copy()
        panel = panel.sort_values(["employee_id", "period"]).drop_duplicates(
            subset=["employee_id", "period"], keep="last"
        )
        return panel.reset_index(drop=True)


    def enrich_absence_with_panel(absence_standardized: pd.DataFrame, panel_dimension: pd.DataFrame) -> pd.DataFrame:
        joined = absence_standardized.merge(
            panel_dimension,
            on=["employee_id", "period"],
            how="left",
        )
        joined["entity"] = joined["panel_entity"].fillna(joined["absence_entity_raw"]).fillna("Unknown")
        joined["country"] = joined["country"].fillna("Unknown")
        joined["job"] = joined["job"].fillna("Unknown")
        return joined


    def aggregate_employee_month_entity_reason(absence_enriched: pd.DataFrame) -> pd.DataFrame:
        """
        KPI_PLAN step 3:
        Aggregate absence by month, entity, reason, employee.
        """
        if absence_enriched.empty:
            return pd.DataFrame(
                columns=[
                    "employee_id",
                    "period",
                    "year",
                    "country",
                    "entity",
                    "job",
                    "reason_group",
                    "reason_detail",
                    "total_absence_days",
                ]
            )

        grouped = (
            absence_enriched.groupby(
                ["employee_id", "period", "year", "country", "entity", "job", "reason_group", "reason_detail"],
                as_index=False,
            )["absence_days"]
            .sum()
            .rename(columns={"absence_days": "total_absence_days"})
            .sort_values(["period", "entity", "employee_id"])
            .reset_index(drop=True)
        )
        return grouped


    def compute_monthly_drag(employee_month_entity_reason: pd.DataFrame, panel_dimension: pd.DataFrame) -> pd.DataFrame:
        """
        KPI_PLAN step 4:
        Convert monthly absence days into lost FTE.
        """
        if employee_month_entity_reason.empty:
            return pd.DataFrame(
                columns=[
                    "period",
                    "year",
                    "total_absence_days",
                    "employees_with_absence",
                    "active_employees",
                    "absence_days_per_employee",
                    "employees_with_absence_share",
                    "lost_fte",
                    "absence_productivity_drag_proxy",
                ]
            )

        absence_monthly = (
            employee_month_entity_reason.groupby(["period", "year"], as_index=False)
            .agg(
                total_absence_days=("total_absence_days", "sum"),
                employees_with_absence=("employee_id", "nunique"),
            )
            .sort_values("period")
            .reset_index(drop=True)
        )

        active_monthly = (
            panel_dimension.groupby("period", as_index=False)["employee_id"]
            .nunique()
            .rename(columns={"employee_id": "active_employees"})
        )

        monthly = absence_monthly.merge(active_monthly, on="period", how="left")
        monthly["active_employees"] = pd.to_numeric(monthly["active_employees"], errors="coerce")
        monthly["absence_days_per_employee"] = _safe_divide(monthly["total_absence_days"], monthly["active_employees"])
        monthly["employees_with_absence_share"] = _safe_divide(
            monthly["employees_with_absence"], monthly["active_employees"]
        )
        monthly["lost_fte"] = monthly["total_absence_days"] / 220.0
        monthly["absence_productivity_drag_proxy"] = _safe_divide(monthly["lost_fte"], monthly["active_employees"])

        return monthly.sort_values("period").reset_index(drop=True)


    def load_finance_reference(finance_path: Path, finance_scope: Scope) -> pd.DataFrame:
        """
        Load yearly HCVA per FTE and average FTE from KPI 1 source.
        """
        table = build_kpi1_table(KPI1Config(file_path=finance_path, scope=finance_scope))
        return table.loc[:, ["year", "avg_fte", "hcva", "hcva_per_fte", "hcroi", "revenue_per_fte"]].copy()


    def compute_yearly_drag(monthly_drag: pd.DataFrame, finance_reference: pd.DataFrame) -> pd.DataFrame:
        """
        KPI_PLAN step 5:
        Compute yearly absence drag and estimated value lost.
        """
        if monthly_drag.empty:
            return pd.DataFrame(
                columns=[
                    "year",
                    "total_absence_days",
                    "lost_fte",
                    "avg_active_employees",
                    "absence_days_per_employee",
                    "avg_fte",
                    "absence_productivity_drag",
                    "hcva_per_fte",
                    "estimated_value_lost",
                    "estimated_value_lost_share_of_hcva",
                ]
            )

        yearly = (
            monthly_drag.groupby("year", as_index=False)
            .agg(
                total_absence_days=("total_absence_days", "sum"),
                lost_fte=("lost_fte", "sum"),
                avg_active_employees=("active_employees", "mean"),
                avg_employees_with_absence=("employees_with_absence", "mean"),
                avg_absence_share=("employees_with_absence_share", "mean"),
                months_observed=("period", "nunique"),
            )
            .sort_values("year")
            .reset_index(drop=True)
        )

        yearly["absence_days_per_employee"] = _safe_divide(yearly["total_absence_days"], yearly["avg_active_employees"])
        yearly = yearly.merge(finance_reference, on="year", how="left")
        yearly["avg_fte_for_drag"] = yearly["avg_fte"].fillna(yearly["avg_active_employees"])
        yearly["absence_productivity_drag"] = _safe_divide(yearly["lost_fte"], yearly["avg_fte_for_drag"])
        yearly["estimated_value_lost"] = yearly["lost_fte"] * yearly["hcva_per_fte"]
        yearly["estimated_value_lost_share_of_hcva"] = _safe_divide(yearly["estimated_value_lost"], yearly["hcva"])

        return yearly


    def compute_entity_year_drag(
        employee_month_entity_reason: pd.DataFrame,
        panel_dimension: pd.DataFrame,
        finance_reference: pd.DataFrame,
    ) -> pd.DataFrame:
        if employee_month_entity_reason.empty:
            return pd.DataFrame(
                columns=[
                    "year",
                    "entity",
                    "total_absence_days",
                    "affected_employees",
                    "lost_fte",
                    "avg_active_employees",
                    "absence_days_per_employee",
                    "absence_drag_proxy",
                    "estimated_value_lost",
                ]
            )

        absence_entity = (
            employee_month_entity_reason.groupby(["year", "entity"], as_index=False)
            .agg(
                total_absence_days=("total_absence_days", "sum"),
                affected_employees=("employee_id", "nunique"),
            )
            .sort_values(["year", "total_absence_days"], ascending=[True, False])
            .reset_index(drop=True)
        )
        absence_entity["lost_fte"] = absence_entity["total_absence_days"] / 220.0

        panel_entity_month = (
            panel_dimension.assign(year=panel_dimension["period"].dt.year)
            .groupby(["year", "period", "panel_entity"], as_index=False)["employee_id"]
            .nunique()
            .rename(columns={"panel_entity": "entity", "employee_id": "active_employees"})
        )
        panel_entity_year = (
            panel_entity_month.groupby(["year", "entity"], as_index=False)["active_employees"]
            .mean()
            .rename(columns={"active_employees": "avg_active_employees"})
        )

        entity = absence_entity.merge(panel_entity_year, on=["year", "entity"], how="left")
        entity["absence_days_per_employee"] = _safe_divide(entity["total_absence_days"], entity["avg_active_employees"])
        entity["absence_drag_proxy"] = _safe_divide(entity["lost_fte"], entity["avg_active_employees"])
        entity = entity.merge(finance_reference[["year", "hcva_per_fte"]], on="year", how="left")
        entity["estimated_value_lost"] = entity["lost_fte"] * entity["hcva_per_fte"]
        return entity.sort_values(["year", "estimated_value_lost"], ascending=[True, False]).reset_index(drop=True)


    def compute_reason_year_drag(
        employee_month_entity_reason: pd.DataFrame,
        finance_reference: pd.DataFrame,
        reason_column: str,
    ) -> pd.DataFrame:
        if employee_month_entity_reason.empty:
            return pd.DataFrame(
                columns=[
                    "year",
                    reason_column,
                    "total_absence_days",
                    "affected_employees",
                    "lost_fte",
                    "estimated_value_lost",
                ]
            )

        grouped = (
            employee_month_entity_reason.groupby(["year", reason_column], as_index=False)
            .agg(
                total_absence_days=("total_absence_days", "sum"),
                affected_employees=("employee_id", "nunique"),
            )
            .sort_values(["year", "total_absence_days"], ascending=[True, False])
            .reset_index(drop=True)
        )
        grouped["lost_fte"] = grouped["total_absence_days"] / 220.0
        grouped = grouped.merge(finance_reference[["year", "hcva_per_fte"]], on="year", how="left")
        grouped["estimated_value_lost"] = grouped["lost_fte"] * grouped["hcva_per_fte"]
        return grouped.sort_values(["year", "estimated_value_lost"], ascending=[True, False]).reset_index(drop=True)


    def build_kpi4_result(config: KPI4Config) -> KPI4Result:
        """
        KPI_PLAN steps 1-5 end-to-end builder for KPI 4.
        """
        absence_standardized = load_absence_standardized(config.absence_paths)
        panel_dimension = load_employee_month_dimension(config.panel_path)
        absence_enriched = enrich_absence_with_panel(absence_standardized, panel_dimension)
        employee_month_entity_reason = aggregate_employee_month_entity_reason(absence_enriched)

        monthly_drag = compute_monthly_drag(employee_month_entity_reason, panel_dimension)
        finance_reference = load_finance_reference(config.finance_path, config.finance_scope)
        yearly_drag = compute_yearly_drag(monthly_drag, finance_reference)
        entity_year_drag = compute_entity_year_drag(employee_month_entity_reason, panel_dimension, finance_reference)
        reason_group_year_drag = compute_reason_year_drag(
            employee_month_entity_reason=employee_month_entity_reason,
            finance_reference=finance_reference,
            reason_column="reason_group",
        )
        reason_detail_year_drag = compute_reason_year_drag(
            employee_month_entity_reason=employee_month_entity_reason,
            finance_reference=finance_reference,
            reason_column="reason_detail",
        )

        latest_year = int(yearly_drag["year"].max()) if not yearly_drag.empty else None
        latest_year_entity_drag = (
            entity_year_drag.loc[entity_year_drag["year"] == latest_year].copy()
            if latest_year is not None
            else entity_year_drag.copy()
        )
        latest_year_reason_group_drag = (
            reason_group_year_drag.loc[reason_group_year_drag["year"] == latest_year].copy()
            if latest_year is not None
            else reason_group_year_drag.copy()
        )
        latest_year_reason_detail_drag = (
            reason_detail_year_drag.loc[reason_detail_year_drag["year"] == latest_year].copy()
            if latest_year is not None
            else reason_detail_year_drag.copy()
        )

        latest_year_entity_drag = latest_year_entity_drag.sort_values(
            ["estimated_value_lost", "total_absence_days"], ascending=[False, False]
        ).reset_index(drop=True)
        latest_year_reason_group_drag = latest_year_reason_group_drag.sort_values(
            ["estimated_value_lost", "total_absence_days"], ascending=[False, False]
        ).reset_index(drop=True)
        latest_year_reason_detail_drag = latest_year_reason_detail_drag.sort_values(
            ["estimated_value_lost", "total_absence_days"], ascending=[False, False]
        ).reset_index(drop=True)

        return KPI4Result(
            absence_standardized_rows=absence_standardized,
            employee_month_entity_reason=employee_month_entity_reason,
            monthly_drag=monthly_drag,
            yearly_drag=yearly_drag,
            entity_year_drag=entity_year_drag,
            reason_group_year_drag=reason_group_year_drag,
            reason_detail_year_drag=reason_detail_year_drag,
            latest_year_entity_drag=latest_year_entity_drag,
            latest_year_reason_group_drag=latest_year_reason_group_drag,
            latest_year_reason_detail_drag=latest_year_reason_detail_drag,
        )


    def summarize_kpi4(result: KPI4Result) -> list[str]:
        """
        Short business interpretation for KPI 4 outputs.
        """
        yearly = result.yearly_drag
        if yearly.empty:
            return ["No KPI 4 records were generated."]

        latest = yearly.sort_values("year").iloc[-1]
        latest_year = int(latest["year"])

        lines = [
            (
                f"Latest year: {latest_year}. Total absence days = {latest['total_absence_days']:,.1f}, "
                f"Lost FTE = {latest['lost_fte']:,.1f}, Absence Productivity Drag = {latest['absence_productivity_drag']:.2%}."
            )
        ]

        if pd.notna(latest.get("estimated_value_lost")):
            lines.append(
                f"Estimated value lost in {latest_year}: {latest['estimated_value_lost']:,.2f} "
                "(Lost FTE multiplied by HCVA per FTE)."
            )

        if len(yearly) >= 2:
            first = yearly.sort_values("year").iloc[0]
            days_delta = latest["total_absence_days"] - first["total_absence_days"]
            lost_fte_delta = latest["lost_fte"] - first["lost_fte"]
            lines.append(
                f"From {int(first['year'])} to {latest_year}, absence days moved by {days_delta:+,.1f} "
                f"and lost FTE moved by {lost_fte_delta:+,.1f}."
            )

        if not result.latest_year_entity_drag.empty:
            top_entity = result.latest_year_entity_drag.iloc[0]
            lines.append(
                f"Top entity drag in {latest_year}: {top_entity['entity']} "
                f"(Lost FTE {top_entity['lost_fte']:,.1f}, value lost {top_entity['estimated_value_lost']:,.2f})."
            )

        if not result.latest_year_reason_group_drag.empty:
            top_reason_group = result.latest_year_reason_group_drag.iloc[0]
            lines.append(
                f"Top absence reason group in {latest_year}: {top_reason_group['reason_group']} "
                f"({top_reason_group['total_absence_days']:,.1f} days)."
            )

        return lines

    return KPI4Config, KPI4Result, build_kpi4_result, summarize_kpi4

KPI4Config, KPI4Result, build_kpi4_result, summarize_kpi4 = _register_kpi4_pipeline()


In [28]:
# KPI 5 Pipeline Code 
def _register_kpi5_pipeline():

    import re
    import unicodedata
    from dataclasses import dataclass
    from pathlib import Path

    import pandas as pd


    DEFAULT_DATA_PATH = Path("Sujet Alberthon/HR Data/Data.xlsx")
    DEFAULT_PANEL_PATH = DEFAULT_DATA_PATH
    DEFAULT_COMPENSATION_PATH = DEFAULT_DATA_PATH
    DEFAULT_EAE_PATH = Path(
        "Sujet Alberthon/HR Data/20250218 - Stats CACEIS EAE EP 18-02-2025 Version Définitive cloture.xlsx"
    )


    @dataclass(frozen=True)
    class KPI5Config:
        panel_path: Path = DEFAULT_PANEL_PATH
        compensation_path: Path = DEFAULT_COMPENSATION_PATH
        eae_path: Path = DEFAULT_EAE_PATH


    @dataclass
    class KPI5Result:
        role_cost_table: pd.DataFrame
        latest_year_table: pd.DataFrame
        country_year_summary: pd.DataFrame
        quadrant_summary: pd.DataFrame
        join_coverage: pd.DataFrame
        efficiency_risk_roles: pd.DataFrame
        strategic_high_value_roles: pd.DataFrame
        retention_risk_roles: pd.DataFrame


    def _normalize_id(values: pd.Series) -> pd.Series:
        cleaned = values.astype(str).str.strip()
        return cleaned.replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA, "None": pd.NA, "NaT": pd.NA})


    def _normalize_text(value: object) -> str:
        if value is None:
            return ""
        if isinstance(value, float) and pd.isna(value):
            return ""
        text = str(value).strip().replace("\xa0", " ")
        text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
        text = text.lower()
        text = re.sub(r"[^a-z0-9]+", " ", text)
        return " ".join(text.split())


    def _extract_year(text: object) -> int | None:
        if text is None:
            return None
        match = re.search(r"(\d{4})", str(text))
        return int(match.group(1)) if match else None


    def _map_eae_country(pays: object, legal_employer: object) -> str | None:
        pays_code = str(pays).strip().upper()
        if pays_code == "FR":
            return "France"
        if pays_code in {"LU", "LUX", "INT"}:
            # In this EAE extract, Luxembourg population is coded as INT.
            return "Luxembourg"

        employer = _normalize_text(legal_employer)
        if "luxembourg" in employer or "luxcellence" in employer:
            return "Luxembourg"
        if "france" in employer:
            return "France"
        return None


    def _weighted_average(values: pd.Series, weights: pd.Series) -> float:
        valid = values.notna() & weights.notna() & (weights >= 0.0)
        if not valid.any():
            return float("nan")
        w = weights[valid]
        v = values[valid]
        w_sum = float(w.sum())
        if w_sum <= 0.0:
            return float(v.mean())
        return float((v * w).sum() / w_sum)


    def load_compensation_long(compensation_path: Path) -> pd.DataFrame:
        """
        KPI_PLAN step 1:
        Transform compensation FR/LU sheets into clean long format.
        """
        sheet_country = {
            "Compensation Data FR": "France",
            "Compensation Data LU": "Luxembourg",
        }
        block_starts = [0, 6, 12]

        frames: list[pd.DataFrame] = []
        for sheet, country in sheet_country.items():
            raw = pd.read_excel(compensation_path, sheet_name=sheet, header=None)

            for start_col in block_starts:
                period_cell = raw.iat[1, start_col + 1] if start_col + 1 < raw.shape[1] else None
                year = _extract_year(period_cell)
                if year is None:
                    continue

                block = raw.iloc[:, start_col : start_col + 4].copy()
                block.columns = ["job", "effectif", "avg_fixed_salary", "avg_variable_salary"]
                block = block.iloc[4:].copy()

                block["job"] = block["job"].astype(str).str.strip()
                block["job_norm"] = block["job"].map(_normalize_text)
                block = block[block["job_norm"] != ""]
                block = block[~block["job_norm"].str.contains(r"^total(?: general)?$", regex=True)]

                block["effectif"] = pd.to_numeric(block["effectif"], errors="coerce")
                block["avg_fixed_salary"] = pd.to_numeric(block["avg_fixed_salary"], errors="coerce")
                block["avg_variable_salary"] = pd.to_numeric(block["avg_variable_salary"], errors="coerce")
                block = block.dropna(subset=["avg_fixed_salary", "avg_variable_salary"], how="all")

                block["country"] = country
                block["year"] = int(year)
                frames.append(block)

        if not frames:
            raise ValueError("No compensation rows were extracted from FR/LU sheets.")

        all_rows = pd.concat(frames, ignore_index=True)

        def aggregate_role(group: pd.DataFrame) -> pd.Series:
            effectif = pd.to_numeric(group["effectif"], errors="coerce").fillna(0.0)
            avg_fixed = _weighted_average(group["avg_fixed_salary"], effectif)
            avg_variable = _weighted_average(group["avg_variable_salary"], effectif)
            if pd.isna(avg_fixed):
                avg_fixed = float(pd.to_numeric(group["avg_fixed_salary"], errors="coerce").mean())
            if pd.isna(avg_variable):
                avg_variable = float(pd.to_numeric(group["avg_variable_salary"], errors="coerce").mean())
            best_label = (
                group.sort_values(["effectif", "avg_fixed_salary"], ascending=[False, False]).iloc[0]["job"]
            )
            return pd.Series(
                {
                    "job": str(best_label).strip(),
                    "effectif": float(effectif.sum()),
                    "avg_fixed_salary": float(avg_fixed),
                    "avg_variable_salary": float(avg_variable),
                }
            )

        grouped_rows: list[dict[str, object]] = []
        for (country, year, job_norm), group in all_rows.groupby(["country", "year", "job_norm"]):
            agg = aggregate_role(group)
            grouped_rows.append(
                {
                    "country": country,
                    "year": int(year),
                    "job_norm": job_norm,
                    "job": agg["job"],
                    "effectif": agg["effectif"],
                    "avg_fixed_salary": agg["avg_fixed_salary"],
                    "avg_variable_salary": agg["avg_variable_salary"],
                }
            )
        grouped = pd.DataFrame(grouped_rows)

        grouped["average_total_compensation"] = grouped["avg_fixed_salary"] + grouped["avg_variable_salary"]
        grouped["variable_pay_ratio"] = grouped["avg_variable_salary"] / grouped["average_total_compensation"]
        grouped["variable_pay_ratio"] = grouped["variable_pay_ratio"].clip(lower=0.0, upper=1.0)

        return grouped.sort_values(["country", "year", "job"]).reset_index(drop=True)


    def build_employee_year_roles(panel_path: Path) -> pd.DataFrame:
        """
        KPI_PLAN step 2 (part):
        Build employee job/country/year base from Data.xlsx > Sheet1.
        """
        raw = pd.read_excel(panel_path, sheet_name="Sheet1")
        use_cols = ["PERIOD", "ID Employee", "COUNTRY_GROUP_LABEL_EN", "POSTE_LABEL_LOCAL", "ENTITY_LABEL_LOCAL"]
        missing_cols = [col for col in use_cols if col not in raw.columns]
        if missing_cols:
            raise ValueError(f"Missing columns in panel source: {missing_cols}")

        panel = raw.loc[:, use_cols].rename(
            columns={
                "PERIOD": "period",
                "ID Employee": "employee_id",
                "COUNTRY_GROUP_LABEL_EN": "country",
                "POSTE_LABEL_LOCAL": "job",
                "ENTITY_LABEL_LOCAL": "entity",
            }
        )
        panel["employee_id"] = _normalize_id(panel["employee_id"])
        panel["period"] = pd.to_datetime(panel["period"], errors="coerce", dayfirst=True)
        panel["year"] = panel["period"].dt.year
        panel["job"] = panel["job"].astype(str).str.strip()
        panel["job_norm"] = panel["job"].map(_normalize_text)

        panel = panel.dropna(subset=["employee_id", "period", "year"]).copy()
        panel = panel[panel["country"].isin(["France", "Luxembourg"])].copy()
        panel = panel[panel["job_norm"] != ""].copy()
        panel["year"] = panel["year"].astype(int)

        ordered = panel.sort_values(["employee_id", "year", "period"])
        base = (
            ordered.groupby(["employee_id", "year"], as_index=False)
            .tail(1)
            .loc[:, ["employee_id", "year", "country", "entity", "job", "job_norm"]]
            .reset_index(drop=True)
        )
        return base


    def aggregate_role_performance(eae_path: Path) -> pd.DataFrame:
        """
        KPI_PLAN step 3:
        Aggregate EAE performance by job/year/country.
        """
        raw = pd.read_excel(eae_path, sheet_name="Database")
        raw.columns = [col.strip() if isinstance(col, str) else col for col in raw.columns]

        needed = ["IUG", "Année", "Libellé emploi", "Note de performance", "Pays", "Nom de l'employeur légal"]
        missing = [col for col in needed if col not in raw.columns]
        if missing:
            raise ValueError(f"Missing columns in EAE source: {missing}")

        perf = raw.loc[:, needed].rename(
            columns={
                "IUG": "employee_id",
                "Année": "year",
                "Libellé emploi": "job",
                "Note de performance": "performance_score",
                "Pays": "pays",
                "Nom de l'employeur légal": "legal_employer",
            }
        )

        perf["employee_id"] = _normalize_id(perf["employee_id"])
        perf["year"] = pd.to_numeric(perf["year"], errors="coerce")
        perf["performance_score"] = pd.to_numeric(perf["performance_score"], errors="coerce")
        perf["country"] = perf.apply(
            lambda row: _map_eae_country(row["pays"], row["legal_employer"]), axis=1
        )
        perf["job"] = perf["job"].astype(str).str.strip()
        perf["job_norm"] = perf["job"].map(_normalize_text)

        perf = perf.dropna(subset=["employee_id", "year", "performance_score", "country"]).copy()
        perf = perf[perf["job_norm"] != ""].copy()
        perf["year"] = perf["year"].astype(int)
        perf = perf[perf["country"].isin(["France", "Luxembourg"])].copy()

        per_employee = (
            perf.groupby(["employee_id", "year", "country", "job_norm"], as_index=False)
            .agg(
                performance_score=("performance_score", "mean"),
                job_label_eae=("job", "first"),
            )
            .sort_values(["employee_id", "year"])
            .reset_index(drop=True)
        )

        role = (
            per_employee.groupby(["country", "year", "job_norm"], as_index=False)
            .agg(
                average_role_performance=("performance_score", "mean"),
                performance_employees=("employee_id", "nunique"),
                performance_records=("performance_score", "count"),
                job_label_eae=("job_label_eae", "first"),
            )
            .sort_values(["country", "year", "job_norm"])
            .reset_index(drop=True)
        )
        return role


    def build_role_cost_table(
        employee_year: pd.DataFrame,
        compensation_long: pd.DataFrame,
        role_performance: pd.DataFrame,
    ) -> tuple[pd.DataFrame, pd.DataFrame]:
        """
        KPI_PLAN steps 2-4:
        Join compensation benchmark to employee job/country/year then compute role cost efficiency metrics.
        """
        employee_joined = employee_year.merge(
            compensation_long[
                [
                    "country",
                    "year",
                    "job_norm",
                    "job",
                    "effectif",
                    "avg_fixed_salary",
                    "avg_variable_salary",
                    "average_total_compensation",
                    "variable_pay_ratio",
                ]
            ],
            on=["country", "year", "job_norm"],
            how="left",
            suffixes=("", "_comp"),
        )

        role_population = (
            employee_joined.groupby(["country", "year", "job_norm"], as_index=False)
            .agg(
                employees_panel=("employee_id", "nunique"),
                entities_panel=("entity", "nunique"),
                job_label_panel=("job", "first"),
                compensation_match_share=("average_total_compensation", lambda s: float(s.notna().mean())),
            )
            .sort_values(["country", "year", "job_norm"])
            .reset_index(drop=True)
        )

        role_table = role_population.merge(
            compensation_long,
            on=["country", "year", "job_norm"],
            how="left",
            suffixes=("_panel", "_comp"),
        )
        role_table = role_table.merge(role_performance, on=["country", "year", "job_norm"], how="left")

        role_table["job"] = role_table["job"].fillna(role_table["job_label_panel"]).fillna(
            role_table["job_label_eae"]
        )

        role_table["median_country_year_compensation"] = role_table.groupby(["country", "year"])[
            "average_total_compensation"
        ].transform("median")
        role_table["role_compensation_index"] = (
            role_table["average_total_compensation"] / role_table["median_country_year_compensation"]
        )

        role_table["performance_score_norm_0_1"] = (role_table["average_role_performance"] / 5.0).clip(
            lower=0.0, upper=1.0
        )
        role_table["median_country_year_role_performance"] = role_table.groupby(["country", "year"])[
            "average_role_performance"
        ].transform("median")
        role_table["normalized_average_role_performance"] = (
            role_table["average_role_performance"] / role_table["median_country_year_role_performance"]
        )

        fallback_norm = role_table["performance_score_norm_0_1"] * 2.0
        role_table["normalized_average_role_performance"] = role_table[
            "normalized_average_role_performance"
        ].fillna(fallback_norm)

        role_table["role_cost_efficiency"] = (
            role_table["normalized_average_role_performance"] / role_table["role_compensation_index"]
        )

        has_comp = role_table["role_compensation_index"].notna()
        has_perf = role_table["normalized_average_role_performance"].notna()
        high_cost = role_table["role_compensation_index"] >= 1.0
        high_perf = role_table["normalized_average_role_performance"] >= 1.0

        role_table["quadrant"] = "Data unavailable"
        role_table.loc[has_comp & has_perf & high_cost & high_perf, "quadrant"] = (
            "High cost + high performance"
        )
        role_table.loc[has_comp & has_perf & high_cost & (~high_perf), "quadrant"] = (
            "High cost + low performance"
        )
        role_table.loc[has_comp & has_perf & (~high_cost) & high_perf, "quadrant"] = (
            "Low cost + high performance"
        )
        role_table.loc[has_comp & has_perf & (~high_cost) & (~high_perf), "quadrant"] = (
            "Low cost + low performance"
        )

        ordered_cols = [
            "country",
            "year",
            "job",
            "job_norm",
            "employees_panel",
            "entities_panel",
            "effectif",
            "avg_fixed_salary",
            "avg_variable_salary",
            "average_total_compensation",
            "variable_pay_ratio",
            "median_country_year_compensation",
            "role_compensation_index",
            "average_role_performance",
            "performance_score_norm_0_1",
            "median_country_year_role_performance",
            "normalized_average_role_performance",
            "role_cost_efficiency",
            "performance_employees",
            "performance_records",
            "compensation_match_share",
            "quadrant",
        ]
        role_table = role_table.loc[:, ordered_cols].sort_values(
            ["year", "country", "role_cost_efficiency"], ascending=[True, True, False]
        )
        return role_table.reset_index(drop=True), employee_joined


    def build_kpi5_result(config: KPI5Config) -> KPI5Result:
        """
        KPI_PLAN steps 1-5 end-to-end builder.
        """
        compensation_long = load_compensation_long(config.compensation_path)
        employee_year = build_employee_year_roles(config.panel_path)
        role_performance = aggregate_role_performance(config.eae_path)

        role_cost_table, employee_joined = build_role_cost_table(
            employee_year=employee_year,
            compensation_long=compensation_long,
            role_performance=role_performance,
        )

        valid_years = role_cost_table.loc[role_cost_table["average_role_performance"].notna(), "year"]
        latest_year = int(valid_years.max()) if not valid_years.empty else int(role_cost_table["year"].max())
        latest = role_cost_table[role_cost_table["year"] == latest_year].copy()
        latest = latest.sort_values(["country", "role_cost_efficiency"], ascending=[True, False]).reset_index(
            drop=True
        )

        country_year_summary = (
            role_cost_table.groupby(["country", "year"], as_index=False)
            .agg(
                roles=("job_norm", "nunique"),
                employees_panel=("employees_panel", "sum"),
                avg_total_compensation=("average_total_compensation", "mean"),
                median_total_compensation=("average_total_compensation", "median"),
                avg_role_compensation_index=("role_compensation_index", "mean"),
                avg_role_performance=("average_role_performance", "mean"),
                avg_normalized_role_performance=("normalized_average_role_performance", "mean"),
                avg_role_cost_efficiency=("role_cost_efficiency", "mean"),
                performance_coverage=("average_role_performance", lambda s: float(s.notna().mean())),
                efficiency_risk_share=("quadrant", lambda s: float((s == "High cost + low performance").mean())),
                retention_risk_share=("quadrant", lambda s: float((s == "Low cost + high performance").mean())),
            )
            .sort_values(["year", "country"])
            .reset_index(drop=True)
        )

        quadrant_summary = (
            latest.groupby(["country", "quadrant"], as_index=False)
            .agg(
                roles=("job_norm", "nunique"),
                employees_panel=("employees_panel", "sum"),
            )
            .sort_values(["country", "roles"], ascending=[True, False])
            .reset_index(drop=True)
        )

        join_coverage = (
            employee_joined.groupby(["country", "year"], as_index=False)
            .agg(
                employee_year_rows=("employee_id", "count"),
                employees=("employee_id", "nunique"),
                panel_jobs=("job_norm", "nunique"),
                compensation_match_share=("average_total_compensation", lambda s: float(s.notna().mean())),
                matched_employee_year_rows=("average_total_compensation", lambda s: int(s.notna().sum())),
            )
            .sort_values(["year", "country"])
            .reset_index(drop=True)
        )

        efficiency_risk_roles = (
            latest[latest["quadrant"] == "High cost + low performance"]
            .sort_values(["role_compensation_index", "employees_panel"], ascending=[False, False])
            .reset_index(drop=True)
        )
        strategic_high_value_roles = (
            latest[latest["quadrant"] == "High cost + high performance"]
            .sort_values(["role_cost_efficiency", "employees_panel"], ascending=[False, False])
            .reset_index(drop=True)
        )
        retention_risk_roles = (
            latest[latest["quadrant"] == "Low cost + high performance"]
            .sort_values(["role_cost_efficiency", "employees_panel"], ascending=[False, False])
            .reset_index(drop=True)
        )

        return KPI5Result(
            role_cost_table=role_cost_table,
            latest_year_table=latest,
            country_year_summary=country_year_summary,
            quadrant_summary=quadrant_summary,
            join_coverage=join_coverage,
            efficiency_risk_roles=efficiency_risk_roles,
            strategic_high_value_roles=strategic_high_value_roles,
            retention_risk_roles=retention_risk_roles,
        )


    def summarize_kpi5(result: KPI5Result) -> list[str]:
        """
        KPI_PLAN step 5:
        Provide business interpretation for role cost-performance alignment.
        """
        latest = result.latest_year_table
        if latest.empty:
            return ["No KPI 5 role records were generated."]

        latest_year = int(latest["year"].max())
        perf_coverage = float(latest["average_role_performance"].notna().mean())
        avg_efficiency = float(latest["role_cost_efficiency"].mean())
        messages = [
            (
                f"Latest year with role performance data: {latest_year}. "
                f"Average role cost efficiency is {avg_efficiency:.3f} with {perf_coverage:.1%} role-level performance coverage."
            )
        ]

        if not result.efficiency_risk_roles.empty:
            top_risk = result.efficiency_risk_roles.iloc[0]
            messages.append(
                "Top efficiency-risk role: "
                f"{top_risk['job']} ({top_risk['country']}), "
                f"cost index {top_risk['role_compensation_index']:.2f}, "
                f"normalized performance {top_risk['normalized_average_role_performance']:.2f}."
            )

        if not result.strategic_high_value_roles.empty:
            top_strategic = result.strategic_high_value_roles.iloc[0]
            messages.append(
                "Top strategic high-value role: "
                f"{top_strategic['job']} ({top_strategic['country']}), "
                f"role cost efficiency {top_strategic['role_cost_efficiency']:.2f}."
            )

        if not result.retention_risk_roles.empty:
            top_retention = result.retention_risk_roles.iloc[0]
            messages.append(
                "Top retention-risk role (high performance at lower relative cost): "
                f"{top_retention['job']} ({top_retention['country']})."
            )

        messages.append(
            "Limitation: compensation input is role-level average benchmark, not individual employee salary."
        )
        return messages

    return KPI5Config, KPI5Result, build_kpi5_result, summarize_kpi5

KPI5Config, KPI5Result, build_kpi5_result, summarize_kpi5 = _register_kpi5_pipeline()


## Run All KPI Pipelines

This cell executes all 5 KPI pipelines using the current project data files.
If it runs without error, your implementation is technically reproducible.


In [29]:
finance_path = Path('Sujet Alberthon/Finance/AlbertSchool_CACEIS_PL-FTE_22-25_Sent.xlsx')

kpi1_table = build_kpi1_table(KPI1Config(file_path=finance_path, scope='group'))
kpi2_result = build_kpi2_result(KPI2Config())
kpi3_result = build_kpi3_result(KPI3Config())
kpi4_result = build_kpi4_result(KPI4Config())
kpi5_result = build_kpi5_result(KPI5Config())

print('Pipelines executed successfully: KPI1 to KPI5')
print('KPI1 rows:', len(kpi1_table))
print('KPI2 risk table rows:', len(kpi2_result.risk_table))
print('KPI3 employee-year rows:', len(kpi3_result.employee_year_table))
print('KPI4 yearly drag rows:', len(kpi4_result.yearly_drag))
print('KPI5 role cost rows:', len(kpi5_result.role_cost_table))


Pipelines executed successfully: KPI1 to KPI5
KPI1 rows: 4
KPI2 risk table rows: 275182
KPI3 employee-year rows: 28452
KPI4 yearly drag rows: 3
KPI5 role cost rows: 964


## KPI 1 - HCVA / HCROI per FTE

What the code does:
- Extracts yearly P&L metrics from `Synthese_PL`.
- Extracts yearly average FTE from `Synthese_ETP`.
- Computes `HCVA`, `HCVA per FTE`, `HCROI`, and `Revenue per FTE`.
- Generates trend interpretation from first year to last year.


In [30]:
kpi1_table


,year,Net Banking Income (PNB),Other Operating Costs,Total Personnel Costs,avg_fte,hcva,hcva_per_fte,hcroi,revenue_per_fte
0,2022,1.249965e+06,435075.76076,460439.98447,3963.56,8.148892e+05,205.595276,1.769805,315.364211
1,2023,1.677332e+06,581401.57132,601963.10454,6370.66,1.095930e+06,172.027706,1.820593,263.290082
2,2024,2.083437e+06,715238.29784,764928.35012,6635.56,1.368198e+06,206.191848,1.788662,313.980535
3,2025,2.100011e+06,650282.29378,790137.79763,6631.47,1.449728e+06,218.613420,1.834779,316.673472


In [31]:
for line in summarize_trend(kpi1_table):
    print('-', line)


- HCVA per FTE moved from 205.60 in 2022 to 218.61 in 2025 (+6.3%).
- HCROI moved from 1.77 to 1.83 (+3.7%).
- Revenue per FTE moved from 315.36 to 316.67 (+0.4%).
- Interpretation: workforce value creation improved relative to workforce cost.


## KPI 2 - Workforce Value-at-Risk

What the code does:
- Builds an employee-month panel from `Data.xlsx`.
- Creates target `left_next_6m` (employee disappears from next 6 months).
- Integrates trailing 12-month absence and training, plus EAE performance score.
- Computes weighted risk score and risk bands (`Low`, `Medium`, `High`).
- Produces risk trends and high-risk entity/job segmentation.


In [32]:
latest_period_kpi2 = kpi2_result.latest_risk_table['period'].max()
print('Latest KPI2 month:', latest_period_kpi2)

risk_band_dist = (
    kpi2_result.latest_risk_table['risk_band']
    .value_counts(dropna=False)
    .rename_axis('risk_band')
    .reset_index(name='employees')
)

risk_band_dist['share'] = risk_band_dist['employees'] / risk_band_dist['employees'].sum()
risk_band_dist


Latest KPI2 month: 2025-12-01 00:00:00


,risk_band,employees,share
0,Low,3532,0.431521
1,Medium,2457,0.300183
2,High,2196,0.268296


In [33]:
kpi2_result.entity_risk.head(10)


,entity,employees,avg_risk_score,high_risk_share
0,CACEIS Canada Asset Servicing,113,0.401289,0.672566
1,CACEIS Bank New York representative office,4,0.366087,0.500000
2,CACEIS FUND ADMINISTRATION SUCURSAL EN ESPAÑA,98,0.349186,0.438776
3,CACEIS Bank Germany Branch,609,0.348488,0.440066
4,"BANCO S3 CACEIS MEXICO, S.A.",109,0.339678,0.321101
5,Santander Caceis Brasil Distribuidora de titul...,278,0.336935,0.312950
6,CACEIS Bank Italy Branch,112,0.335826,0.303571
7,CACEIS Bank Belgium Branch,80,0.335454,0.312500
8,CACEIS Fund Administration,625,0.321816,0.385600
9,CACEIS Fund Admininstration Jersey (CI) Limited,12,0.321618,0.250000


In [34]:
kpi2_result.job_risk.head(10)


,job,employees,avg_risk_score,high_risk_share
0,Compliance Officer (Junior),1,0.501700,1.000000
1,HEAD OF CLIENT MANAGEMENT,1,0.470450,1.000000
2,Intern,16,0.461797,0.937500
3,Communication Assistant,1,0.449616,1.000000
4,Apprentice,8,0.448010,0.750000
5,INTERN,358,0.447888,0.924581
6,Learning & Development Coordinator,2,0.444016,1.000000
7,IT Project Manager,1,0.438208,1.000000
8,Working Student,22,0.430937,0.863636
9,ANALYST,1,0.428629,1.000000


In [35]:
kpi2_result.risk_trend.tail(12)


,period,avg_risk_score,high_risk_share,observed_leave_rate,employees
24,2025-01-01,0.318653,0.308479,0.010969,8114
25,2025-02-01,0.317172,0.310714,0.013916,8120
26,2025-03-01,0.315471,0.303519,0.013041,8128
27,2025-04-01,0.315511,0.300815,0.009488,8221
28,2025-05-01,0.313820,0.299539,0.011036,8246
29,2025-06-01,0.313193,0.297038,0.028301,8339
30,2025-07-01,0.309261,0.287872,NaN,8344
31,2025-08-01,0.308199,0.285076,NaN,8275
32,2025-09-01,0.307370,0.281588,NaN,8239
33,2025-10-01,0.307750,0.278951,NaN,8195


In [36]:
for line in summarize_kpi2(kpi2_result):
    print('-', line)


- Latest month in panel: 2025-12. Average risk score is 0.304 with 26.8% employees in High risk.
- Highest-risk entity (by average score): CACEIS Canada Asset Servicing.
- Highest-risk job family (by average score): Compliance Officer (Junior).
- Average risk score trend moved from 0.304 to 0.304 (-0.000).


## KPI 3 - Learning-to-Performance Index

What the code does:
- Aggregates training records by employee-year.
- Converts quick/cold reviews into normalized learning signal.
- Integrates annual EAE performance score.
- Computes Learning-to-Performance Index from weighted components.
- Compares trained vs non-trained employees and calculates correlations.


In [37]:
kpi3_result.yearly_summary


,year,employees,trained_share,certified_share,avg_training_hours,avg_training_count,avg_quick_review_score,avg_cold_impact_score,avg_learning_index,avg_performance_score,performance_coverage
0,2023,9188,0.164562,0.000000,4.293740,0.499565,4.306944,0.683550,0.050712,NaN,0.000000
1,2024,9541,0.165706,0.010586,3.562657,0.423121,4.531446,0.726626,0.050351,3.39081,0.363798
2,2025,9723,0.158593,0.002160,3.611378,0.369228,4.524726,0.754803,0.045936,NaN,0.000000


In [38]:
kpi3_result.trained_vs_non_trained


,year,trained_employees,non_trained_employees,trained_avg_performance,non_trained_avg_performance,trained_avg_learning_index,non_trained_avg_learning_index,trained_avg_training_hours,non_trained_avg_training_hours,trained_avg_cold_impact_score,non_trained_avg_cold_impact_score,performance_gap_trained_minus_non_trained,learning_index_gap_trained_minus_non_trained
0,2023,1512,7676,NaN,NaN,0.308063,0.000020,26.091852,0.0,0.683550,NaN,NaN,0.308043
1,2024,1581,7960,3.357475,3.41055,0.303629,0.000046,21.499880,0.0,0.726626,NaN,-0.053076,0.303584
2,2025,1542,8181,NaN,NaN,0.289526,0.000023,22.771355,0.0,0.754803,NaN,NaN,0.289503


In [39]:
corr_training_performance = kpi3_result.scatter_training_performance['training_hours'].corr(
    kpi3_result.scatter_training_performance['performance_score']
)
corr_training_impact = kpi3_result.scatter_training_impact['training_hours'].corr(
    kpi3_result.scatter_training_impact['cold_impact_score']
)

print('Correlation training_hours vs performance_score:', round(corr_training_performance, 3))
print('Correlation training_hours vs cold_impact_score:', round(corr_training_impact, 3))


Correlation training_hours vs performance_score: 0.116
Correlation training_hours vs cold_impact_score: 0.19


In [40]:
for line in summarize_kpi3(kpi3_result):
    print('-', line)


- Latest year: 2025. Average Learning-to-Performance Index is 0.046, with 15.9% employees trained.
- Performance gap (trained - non-trained) in latest year with EAE coverage (2024): -0.053 points.
- Correlation between training hours and performance score: +0.116.
- Correlation between training hours and cold impact score: +0.190.
- Highest learning index entity in latest year: CACEIS.


## KPI 4 - Absence Productivity Drag

What the code does:
- Loads and standardizes 2023/2024/2025 absence files.
- Aggregates by employee/month/entity/reason.
- Converts absence days into lost FTE (`days / 220`).
- Estimates value lost by joining with KPI1 `HCVA per FTE`.
- Produces yearly/entity/reason drag outputs and trends.


In [41]:
kpi4_result.yearly_drag


,year,total_absence_days,lost_fte,avg_active_employees,avg_employees_with_absence,avg_absence_share,months_observed,absence_days_per_employee,avg_fte,hcva,hcva_per_fte,hcroi,revenue_per_fte,avg_fte_for_drag,absence_productivity_drag,estimated_value_lost,estimated_value_lost_share_of_hcva
0,2023,207434.5,942.884091,6607.666667,1751.833333,0.273178,12,31.393003,6370.66,1.095930e+06,172.027706,1.820593,263.290082,6370.66,0.148004,162202.187033,0.148004
1,2024,103375.5,469.888636,8107.166667,1515.583333,0.186901,12,12.751126,6635.56,1.368198e+06,206.191848,1.788662,313.980535,6635.56,0.070814,96887.206497,0.070814
2,2025,98744.5,448.838636,8217.000000,1442.666667,0.175475,12,12.017099,6631.47,1.449728e+06,218.613420,1.834779,316.673472,6631.47,0.067683,98122.149385,0.067683


In [42]:
kpi4_result.latest_year_entity_drag.head(10)


,year,entity,total_absence_days,affected_employees,lost_fte,avg_active_employees,absence_days_per_employee,absence_drag_proxy,hcva_per_fte,estimated_value_lost
0,2025,CACEIS Bank,66293.0,1530,301.331818,1555.750000,42.611602,0.193689,218.61342,65875.179369
1,2025,CACEIS Fund Administration,29779.0,715,135.359091,673.083333,44.242664,0.201103,218.61342,29591.313810
2,2025,CACEIS,2645.5,81,12.025000,88.166667,30.005671,0.136389,218.61342,2628.826377
3,2025,CREDIT AGRICOLE S.A.,27.0,1,0.122727,1.090909,24.750000,0.112500,218.61342,26.829829


In [43]:
kpi4_result.latest_year_reason_group_drag.head(10)


,year,reason_group,total_absence_days,affected_employees,lost_fte,hcva_per_fte,estimated_value_lost
0,2025,Congés,70174.5,2222,318.975000,218.61342,69732.215688
1,2025,Maladie,16183.0,808,73.559091,218.61342,16081.004446
2,2025,Maternité et paternité,7132.5,106,32.420455,218.61342,7087.546451
3,2025,Absences non suivi,2161.0,223,9.822727,218.61342,2147.380004
4,2025,Légal / Conventionnel Familial,1417.5,490,6.443182,218.61342,1408.566014
5,2025,Légal / Conventionnel Autre,843.5,82,3.834091,218.61342,838.183727
6,2025,Absences Autres motifs,404.5,17,1.838636,218.61342,401.950584
7,2025,Accident,384.0,15,1.745455,218.61342,381.579788
8,2025,Absence non Autorisée,44.0,3,0.200000,218.61342,43.722684


In [44]:
for line in summarize_kpi4(kpi4_result):
    print('-', line)


- Latest year: 2025. Total absence days = 98,744.5, Lost FTE = 448.8, Absence Productivity Drag = 6.77%.
- Estimated value lost in 2025: 98,122.15 (Lost FTE multiplied by HCVA per FTE).
- From 2023 to 2025, absence days moved by -108,690.0 and lost FTE moved by -494.0.
- Top entity drag in 2025: CACEIS Bank (Lost FTE 301.3, value lost 65,875.18).
- Top absence reason group in 2025: Congés (70,174.5 days).


## KPI 5 - Role Cost Efficiency Index

What the code does:
- Reshapes compensation benchmarks from FR/LU sheets into long format.
- Aligns panel roles with compensation and EAE role-level performance.
- Calculates role compensation index and role cost efficiency score.
- Assigns each role to a cost-performance quadrant.
- Identifies efficiency-risk, strategic high-value, and retention-risk roles.


In [45]:
kpi5_result.country_year_summary


,country,year,roles,employees_panel,avg_total_compensation,median_total_compensation,avg_role_compensation_index,avg_role_performance,avg_normalized_role_performance,avg_role_cost_efficiency,performance_coverage,efficiency_risk_share,retention_risk_share
0,France,2023,209,2720,75148.052512,62350.220000,1.205257,NaN,NaN,NaN,0.000000,0.000000,0.000000
1,Luxembourg,2023,136,2535,102510.475619,80540.320208,1.272785,NaN,NaN,NaN,0.000000,0.000000,0.000000
2,France,2024,156,2709,95794.176759,66810.271233,1.433824,3.387710,1.004834,0.947534,0.897436,0.134615,0.147436
3,Luxembourg,2024,164,2485,122138.116974,97846.114235,1.248267,3.457388,1.007006,0.972477,0.823171,0.134146,0.134146
4,France,2025,150,2782,99664.455162,69055.650000,1.443248,NaN,NaN,NaN,0.000000,0.000000,0.000000
5,Luxembourg,2025,149,2509,121445.121680,96974.641000,1.252339,NaN,NaN,NaN,0.000000,0.000000,0.000000


In [46]:
kpi5_result.quadrant_summary


,country,quadrant,roles,employees_panel
0,France,High cost + high performance,44,477
1,France,Low cost + low performance,43,1493
2,France,Data unavailable,25,398
3,France,Low cost + high performance,23,250
4,France,High cost + low performance,21,91
5,Luxembourg,Low cost + low performance,41,915
6,Luxembourg,High cost + high performance,40,486
7,Luxembourg,Data unavailable,39,484
8,Luxembourg,High cost + low performance,22,186
9,Luxembourg,Low cost + high performance,22,414


In [47]:
kpi5_result.latest_year_table.head(15)


,country,year,job,job_norm,employees_panel,entities_panel,effectif,avg_fixed_salary,avg_variable_salary,average_total_compensation,variable_pay_ratio,median_country_year_compensation,role_compensation_index,average_role_performance,performance_score_norm_0_1,median_country_year_role_performance,normalized_average_role_performance,role_cost_efficiency,performance_employees,performance_records,compensation_match_share,quadrant
0,France,2024,PARALEGAL OFFICER,paralegal officer,7,2,6.0,42316.546667,1666.666667,43983.213333,0.037893,66810.271233,0.658330,3.666667,0.733333,3.371414,1.087575,1.652021,6.0,6.0,1.0,Low cost + high performance
1,France,2024,FUND ACCOUNTANT,fund accountant,283,2,236.0,41901.438093,1929.478814,43830.916907,0.044021,66810.271233,0.656051,3.200957,0.640191,3.371414,0.949441,1.447206,209.0,209.0,1.0,Low cost + low performance
2,France,2024,USER MANAGEMENT OFFICER,user management officer,9,2,8.0,47205.845000,3275.000000,50480.845000,0.064876,66810.271233,0.755585,3.666667,0.733333,3.371414,1.087575,1.439382,6.0,6.0,1.0,Low cost + high performance
3,France,2024,FINANCING ANALYST MIDDLE OFFICE - PERES,financing analyst middle office peres,3,1,2.0,44750.060000,3750.000000,48500.060000,0.077319,66810.271233,0.725937,3.500000,0.700000,3.371414,1.038140,1.430069,2.0,2.0,1.0,Low cost + high performance
4,France,2024,UCITS CONTROL OFFICER,ucits control officer,9,1,6.0,47593.720000,3900.000000,51493.720000,0.075737,66810.271233,0.770746,3.714286,0.742857,3.371414,1.101700,1.429395,7.0,7.0,1.0,Low cost + high performance
5,France,2024,ASSETS CONTROLLER,assets controller,13,1,15.0,46826.016000,2253.333333,49079.349333,0.045912,66810.271233,0.734608,3.461538,0.692308,3.371414,1.026732,1.397660,13.0,13.0,1.0,Low cost + high performance
6,France,2024,BUSINESS ENGINEERING SUPPORT OFFICER,business engineering support officer,1,1,1.0,51700.160000,5500.000000,57200.160000,0.096154,66810.271233,0.856158,4.000000,0.800000,3.371414,1.186446,1.385779,1.0,1.0,1.0,Low cost + high performance
7,France,2024,CLIENT OPERATIONS OFFICER,client operations officer,413,4,309.0,45759.611446,2310.352751,48069.964196,0.048062,66810.271233,0.719500,3.355072,0.671014,3.371414,0.995153,1.383118,276.0,276.0,1.0,Low cost + low performance
8,France,2024,DATA OFFICER,data officer,77,2,55.0,44761.269818,2601.818182,47363.088000,0.054933,66810.271233,0.708919,3.285714,0.657143,3.371414,0.974581,1.374741,56.0,56.0,1.0,Low cost + low performance
9,France,2024,DATA CONTROLLER,data controller,13,2,11.0,45656.130909,2527.272727,48183.403636,0.052451,66810.271233,0.721198,3.333333,0.666667,3.371414,0.988705,1.370921,9.0,9.0,1.0,Low cost + low performance


In [48]:
for line in summarize_kpi5(kpi5_result):
    print('-', line)


- Latest year with role performance data: 2024. Average role cost efficiency is 0.960 with 85.9% role-level performance coverage.
- Top efficiency-risk role: DEPUTY CHIEF EXECUTIVE OFFICER (France), cost index 9.95, normalized performance 0.89.
- Top strategic high-value role: GROUP DATA PROJECT MANAGER (France), role cost efficiency 1.13.
- Top retention-risk role (high performance at lower relative cost): INVESTOR - CLIENT SERVICES OFFICER (Luxembourg).
- Limitation: compensation input is role-level average benchmark, not individual employee salary.


## Preliminary Findings Snapshot

These findings are extracted directly from current pipeline outputs.


In [49]:
snapshot = {
    'KPI1_latest_hcva_per_fte': float(kpi1_table.sort_values('year').iloc[-1]['hcva_per_fte']),
    'KPI2_latest_avg_risk_score': float(kpi2_result.latest_risk_table['risk_score'].mean()),
    'KPI2_latest_high_risk_share': float((kpi2_result.latest_risk_table['risk_band'] == 'High').mean()),
    'KPI3_latest_avg_learning_index': float(kpi3_result.latest_year_table['learning_to_performance_index'].mean()),
    'KPI3_latest_trained_share': float(kpi3_result.latest_year_table['trained_flag'].mean()),
    'KPI4_latest_absence_drag': float(kpi4_result.yearly_drag.sort_values('year').iloc[-1]['absence_productivity_drag']),
    'KPI5_latest_avg_role_efficiency': float(kpi5_result.latest_year_table['role_cost_efficiency'].mean()),
}

pd.DataFrame([snapshot]).T.rename(columns={0: 'value'})


,value
KPI1_latest_hcva_per_fte,218.613420
KPI2_latest_avg_risk_score,0.303852
KPI2_latest_high_risk_share,0.268296
KPI3_latest_avg_learning_index,0.045936
KPI3_latest_trained_share,0.158593
KPI4_latest_absence_drag,0.067683
KPI5_latest_avg_role_efficiency,0.959713
